In [56]:
import json
with open("cases_all_updated.json", encoding="utf-8") as f:
        backup_cases_all = json.load(f)
backup_cases_all.keys()

dict_keys(['[1998] HCA 28', '[2003] HCA 2', '[2003] HCA 22', '[2001] HCA 30', '[2009] HCA 27', '[2005] HCA 25', '[2010] HCA 16', '[1998] HCA 11', '[2000] HCA 63', '[2006] HCA 63', '[1999] HCA 14', '[2001] HCA 17', '[2000] HCA 54', '[1998] HCA 57', '[2010] HCA 45', '[2004] HCA 52', '[2009] HCA 41', '[2001] HCA 64', '[1999] HCA 21', '[2000] HCA 57', '[2011] HCA 39', '[2000] HCA 47', '[2010] HCA 28', '[2007] HCA 22', '[2010] HCA 1', '[2005] HCA 24', '[2003] HCA 6', '[2000] HCA 48', '[2010] HCA 4', '[1998] HCA 67', '[1999] HCA 54', '[2000] HCA 40', '[2008] HCA 31', '[1999] HCA 29', '[2011] HCA 49', '[2011] HCA 4', '[2002] HCA 11', '[2001] HCA 63', '[2004] HCA 35', '[2002] HCA 53', '[2006] HCA 27', '[2000] HCA 41', '[2002] HCA 36', '[1999] HCA 27', '[1998] HCA 27', '[1999] HCA 26', '[1999] HCA 66', '[2001] HCA 22', '[1999] HCA 67', '[2002] HCA 35', '[2011] HCA 13', '[2010] HCA 41', '[2011] HCA 48', '[2006] HCA 46', '[1998] HCA 55', '[1999] HCA 9', '[2011] HCA 21', '[1998] HCA 30', '[2007] H

In [6]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import networkx as nx
import numpy as np
from collections import defaultdict
from itertools import combinations

# --- Load & compute ---
data = backup_cases_all


co_authored = defaultdict(int)
judge_totals = defaultdict(int)

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            judge_totals[name] += 1
        for a, b in combinations(sorted(names), 2):
            co_authored[(a, b)] += 1

JUDGES = list(judge_totals.keys())

def build_graph(weight_fn):
    G = nx.Graph()
    G.add_nodes_from(JUDGES)
    for (a, b), count in co_authored.items():
        if a in JUDGES and b in JUDGES:
            w = weight_fn(a, b, count)
            if w > 0:
                G.add_edge(a, b, weight=w, raw=count)
    return G

def draw_network(G, title, filename):
    fig, ax = plt.subplots(figsize=(12, 9))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.axis('off')

    pos = nx.spring_layout(G, weight='weight', seed=42, k=2.8)

    weights = [G[u][v]['weight'] for u, v in G.edges()]
    w_min, w_max = min(weights), max(weights)

    # Draw edges — grey, varying width and alpha
    for u, v in G.edges():
        w = G[u][v]['weight']
        t = (w - w_min) / (w_max - w_min) if w_max > w_min else 1.0
        alpha = 0.15 + t * 0.75
        lw    = 0.4 + t * 5.5
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        ax.plot([x0, x1], [y0, y1],
                color='#666666', alpha=alpha, linewidth=lw,
                zorder=1, solid_capstyle='round')

    # Node size proportional to total judgments
    max_total = max(judge_totals[j] for j in JUDGES)
    for judge in JUDGES:
        x, y = pos[judge]
        size = 300 + 1400 * (judge_totals[judge] / max_total)
        ax.scatter(x, y, s=size, color='#333333', zorder=3, edgecolors='#888888', linewidths=1.5)
        # Label offset upward from node centre
        offset = 0.055 + 0.04 * (judge_totals[judge] / max_total)
        label = f"{judge.capitalize()} ({judge_totals[judge]})"
        ax.text(x, y + offset, label,
                ha='center', va='bottom', fontsize=9,
                color='#111111', fontweight='bold', zorder=5)

    ax.set_title(title, fontsize=13, fontweight='bold', pad=14, color='#111111')

    # Legend
    for label, t in [('Weak', 0.05), ('Medium', 0.5), ('Strong', 1.0)]:
        lw    = 0.4 + t * 5.5
        alpha = 0.15 + t * 0.75
        ax.plot([], [], color='#666666', alpha=alpha, linewidth=lw, label=label)
    leg = ax.legend(loc='lower left', frameon=True, fontsize=9,
                    facecolor='white', edgecolor='#aaaaaa',
                    title='Connection strength', title_fontsize=9)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

# Graph 1 — raw
G1 = build_graph(lambda a, b, count: count)
draw_network(G1,
    title="Judge Co-authorship Network (raw count)",
    filename='images/judge_network_raw.png')

# Graph 2 — normalised: 2*count / (total_A + total_B)
def norm_weight(a, b, count):
    return (2 * count) / (judge_totals[a] + judge_totals[b])

G2 = build_graph(norm_weight)
draw_network(G2,
    title="Judge Co-authorship Network (normalised by individual output)",
    filename='images/judge_network_normalised.png')

Saved: images/judge_network_raw.png
Saved: images/judge_network_normalised.png


In [ ]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict

data = backup_cases_all

judge_totals = defaultdict(int)
judge_collab = defaultdict(int)

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            judge_totals[name] += 1
            if len(names) > 1:
                judge_collab[name] += 1

judges   = sorted(judge_totals.keys())
totals   = [judge_totals[j] for j in judges]
collabs  = [judge_collab[j] for j in judges]
ratios   = [judge_collab[j] / judge_totals[j] for j in judges]

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

ax.scatter(totals, ratios, s=80, color='#333333', zorder=3)

# Label each point
for judge, x, y in zip(judges, totals, ratios):
    ax.annotate(
        f"{judge.capitalize()}\n({judge_collab[judge]}/{judge_totals[judge]})",
        xy=(x, y),
        xytext=(6, 4),
        textcoords='offset points',
        fontsize=8.5,
        color='#222222',
    )

ax.set_xlabel('Total judgments written', fontsize=12)
ax.set_ylabel('Collaboration rate\n(joint judgments / total judgments)', fontsize=12)
ax.set_title('Judge Collaboration Rate vs Total Output', fontsize=14, fontweight='bold', pad=14)
ax.set_ylim(-0.05, 1.05)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('images/collab_scatter.png', dpi=150, bbox_inches='tight', facecolor='white')
print("Saved: collab_scatter.png")

Saved: collab_scatter.png


In [8]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
from collections import defaultdict, Counter


data = backup_cases_all

together_raw      = defaultdict(int)
together_majority = defaultdict(int)
together_alone    = defaultdict(int)
tag_totals        = defaultdict(int)
gh_judgment_sizes = []

for case in data.values():
    tags       = case['law_areas']
    bench_size = len(case.get('bench', []))
    for tag in tags:
        tag_totals[tag] += 1

    gh_judgment = None
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        if 'GUMMOW' in names and 'HAYNE' in names:
            gh_judgment = names
            break

    if gh_judgment:
        n_authors  = len(gh_judgment)
        is_majority = n_authors > bench_size / 2 if bench_size > 0 else False
        is_alone    = n_authors == 2
        for tag in tags:
            together_raw[tag] += 1
            if is_majority:
                together_majority[tag] += 1
            if is_alone:
                together_alone[tag] += 1
        gh_judgment_sizes.append(n_authors)

# Sort by raw count descending
tags_sorted  = sorted(tag_totals.keys(), key=lambda t: -together_raw.get(t, 0))
raw          = [together_raw.get(t, 0)      for t in tags_sorted]
normed       = [together_raw.get(t, 0) / tag_totals[t] for t in tags_sorted]
maj_of_tag   = [together_majority.get(t, 0) / tag_totals[t] for t in tags_sorted]
alone_of_tag = [together_alone.get(t, 0)    / tag_totals[t] for t in tags_sorted]
labels       = [t.replace(' & ', '\n& ').replace(', Planning', ',\nPlanning') for t in tags_sorted]
x            = np.arange(len(tags_sorted))

BAR_COLOR  = '#333333'
MAJ_COLOR  = '#cc3333'
ALONE_COLOR = '#3366cc'

# ══════════════════════════════════════════════════════════
# GRAPHS 1, 2 & 3 — stacked subplots
# ══════════════════════════════════════════════════════════
fig, axes = plt.subplots(3, 1, figsize=(16, 15), sharex=True,
                         gridspec_kw={'hspace': 0.12})
fig.patch.set_facecolor('white')

# --- Graph 1: raw counts ---
ax1 = axes[0]
ax1.set_facecolor('#f8f8f8')
ax1.bar(x, raw, color=BAR_COLOR, edgecolor='white', linewidth=0.6, zorder=3)
for i, v in enumerate(raw):
    if v > 0:
        ax1.text(i, v + 0.4, str(v), ha='center', va='bottom', fontsize=8, color='#222')
# n= labels inside top of each bar (or just below bar top)
for i, tag in enumerate(tags_sorted):
    ax1.text(i, 0.8, f'n={tag_totals[tag]}', ha='center', va='bottom',
             fontsize=6.5, color='#aaa', zorder=4)
ax1.set_ylabel('Cases written together', fontsize=11)
ax1.set_title('Gummow & Hayne — Cases co-authored by tag (raw)', fontsize=13, fontweight='bold', pad=10)
ax1.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax1.set_axisbelow(True)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Graph 2: normalised + majority overlay ---
ax2 = axes[1]
ax2.set_facecolor('#f8f8f8')
ax2.bar(x, normed,     color=BAR_COLOR, edgecolor='white', linewidth=0.6, zorder=3, label='Co-authorship rate')
ax2.bar(x, maj_of_tag, color=MAJ_COLOR, edgecolor='white', linewidth=0.6, zorder=4, alpha=0.75,
        label='Of which: majority judgment')
for i, v in enumerate(normed):
    if v > 0:
        ax2.text(i, v + 0.005, f'{v:.0%}', ha='center', va='bottom', fontsize=7.5, color='#222')
ax2.set_ylabel('Share of tag cases', fontsize=11)
ax2.set_title('Gummow & Hayne — Co-authorship rate by tag (normalised), with majority overlay',
              fontsize=13, fontweight='bold', pad=10)
ax2.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax2.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax2.set_axisbelow(True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.legend(fontsize=9, frameon=True, facecolor='white', edgecolor='#aaa')

# --- Graph 3: normalised + alone overlay ---
ax3 = axes[2]
ax3.set_facecolor('#f8f8f8')
ax3.bar(x, normed,       color=BAR_COLOR,  edgecolor='white', linewidth=0.6, zorder=3, label='Co-authorship rate')
ax3.bar(x, alone_of_tag, color=ALONE_COLOR, edgecolor='white', linewidth=0.6, zorder=4, alpha=0.75,
        label='Of which: Gummow & Hayne alone')
for i, v in enumerate(normed):
    if v > 0:
        ax3.text(i, v + 0.005, f'{v:.0%}', ha='center', va='bottom', fontsize=7.5, color='#222')
ax3.set_ylabel('Share of tag cases', fontsize=11)
ax3.set_title('Gummow & Hayne — Co-authorship rate by tag (normalised), with alone overlay',
              fontsize=13, fontweight='bold', pad=10)
ax3.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax3.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax3.set_axisbelow(True)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.legend(fontsize=9, frameon=True, facecolor='white', edgecolor='#aaa')

ax3.set_xticks(x)
ax3.set_xticklabels(labels, rotation=40, ha='right', fontsize=8.5)

plt.tight_layout()
plt.savefig('images/gummow_hayne_by_tag.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: gummow_hayne_by_tag.png")

# ══════════════════════════════════════════════════════════
# JUDGMENT SIZE graph (unchanged)
# ══════════════════════════════════════════════════════════
size_counts = Counter(gh_judgment_sizes)
sizes  = sorted(size_counts.keys())
counts = [size_counts[s] for s in sizes]

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')
bars = ax.bar(sizes, counts, color=BAR_COLOR, edgecolor='white', linewidth=0.8, zorder=3, width=0.6)
for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=10, color='#222')
ax.set_xlabel('Total judges in the Gummow–Hayne judgment', fontsize=11)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_title('Gummow & Hayne — How many judges did they write with?',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(sizes)
ax.set_xticklabels([
    f'{s}\n(G+H {"alone" if s == 2 else f"+ {s-2} other{"s" if s-2 > 1 else ""}"})'
    for s in sizes
], fontsize=9)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('images/gummow_hayne_judgment_size.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: gummow_hayne_judgment_size.png")

/var/folders/4k/yfp_3j857vq5fj717924mjfm0000gn/T/ipykernel_69860/4248368824.py:121: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved: gummow_hayne_by_tag.png
Saved: gummow_hayne_judgment_size.png


In [9]:
for case in backup_cases_all.values():
    judgments = case['judgmentAuthors']
    sorted_j = sorted(judgments, key=lambda j: j.get('totalCitations', 0), reverse=True)
    top_citations = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
 
    for rank, j in enumerate(sorted_j, start=1):
        cites = j.get('totalCitations', 0)
        j['citationRank'] = rank
        j['relativeCitation'] = round(cites / top_citations, 4) if top_citations > 0 else 0
 
    case['judgmentAuthors'] = sorted_j

In [12]:
with open("cases_all_updated.json", "w", encoding="utf-8") as f:
    json.dump(backup_cases_all, f, indent=2, ensure_ascii=False)
    
with open("cases_all_updated.json", encoding="utf-8") as f:
        backup_cases_all = json.load(f)
backup_cases_all.keys()

dict_keys(['[1998] HCA 28', '[2003] HCA 2', '[2003] HCA 22', '[2001] HCA 30', '[2009] HCA 27', '[2005] HCA 25', '[2010] HCA 16', '[1998] HCA 11', '[2000] HCA 63', '[2006] HCA 63', '[1999] HCA 14', '[2001] HCA 17', '[2000] HCA 54', '[1998] HCA 57', '[2010] HCA 45', '[2004] HCA 52', '[2009] HCA 41', '[2001] HCA 64', '[1999] HCA 21', '[2000] HCA 57', '[2011] HCA 39', '[2000] HCA 47', '[2010] HCA 28', '[2007] HCA 22', '[2010] HCA 1', '[2005] HCA 24', '[2003] HCA 6', '[2000] HCA 48', '[2010] HCA 4', '[1998] HCA 67', '[1999] HCA 54', '[2000] HCA 40', '[2008] HCA 31', '[1999] HCA 29', '[2011] HCA 49', '[2011] HCA 4', '[2002] HCA 11', '[2001] HCA 63', '[2004] HCA 35', '[2002] HCA 53', '[2006] HCA 27', '[2000] HCA 41', '[2002] HCA 36', '[1999] HCA 27', '[1998] HCA 27', '[1999] HCA 26', '[1999] HCA 66', '[2001] HCA 22', '[1999] HCA 67', '[2002] HCA 35', '[2011] HCA 13', '[2010] HCA 41', '[2011] HCA 48', '[2006] HCA 46', '[1998] HCA 55', '[1999] HCA 9', '[2011] HCA 21', '[1998] HCA 30', '[2007] H

In [13]:
print(backup_cases_all["[2001] HCA 50"])

{'citation': '[2001] HCA 50', 'catchwords': 'Smith v The Queen Criminal law – Evidence – Relevance – Identification – Evidence of recognition by police officers of the accused in photographs from bank security cameras – Police officers in no better position than jury to compare appearance of accused with photographs – Evidence Act 1995 (NSW), s 55 – Whether evidence could rationally affect the assessment by jury of probability of the existence of a fact in issue – Whether admissible as opinion evidence. Evidence – Relevance – Opinion or fact evidence – Evidence Act 1995 (NSW), s 55 – Evidence of recognition by police officers of accused in photographs from bank security cameras – Whether relevant – If relevant whether excluded as opinion evidence. Evidence Act 1995 (NSW), ss 55, 76.', 'bench': ['GLEESON', 'GAUDRON', 'GUMMOW', 'KIRBY', 'HAYNE'], 'judgmentAuthors': [{'name': 'GLEESON, GAUDRON, GUMMOW, HAYNE', 'type': 'Separate judgment', 'totalCitations': 190, 'paragraphs': '1-17', 'cita

In [14]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

# Add relativeCitation on the fly if not present
for case in data.values():
    judgments = case['judgmentAuthors']
    sorted_j  = sorted(judgments, key=lambda j: j.get('totalCitations', 0), reverse=True)
    top       = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
    for rank, j in enumerate(sorted_j, start=1):
        j['citationRank']     = rank
        j['relativeCitation'] = round(j.get('totalCitations', 0) / top, 4) if top > 0 else 0
    case['judgmentAuthors'] = sorted_j

gummow_together, gummow_alone = [], []
hayne_together,  hayne_alone  = [], []
both_together,   both_alone   = [], []

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        has_g = 'GUMMOW' in names
        has_h = 'HAYNE'  in names
        rc    = j['relativeCitation']

        if has_g and has_h:
            both_together.append(rc)
            gummow_together.append(rc)
            hayne_together.append(rc)
        elif has_g:
            gummow_alone.append(rc)
        elif has_h:
            hayne_alone.append(rc)

DARK   = '#333333'
LIGHT  = '#aaaaaa'
RED    = '#cc3333'

def add_stats(ax, data, pos, color):
    """Annotate median and n beneath each box."""
    ax.text(pos, -0.07, f'n={len(data)}', ha='center', va='top',
            fontsize=8, color='#666', transform=ax.get_xaxis_transform())
    ax.text(pos, -0.12, f'med={np.median(data):.2f}', ha='center', va='top',
            fontsize=8, color=color, fontweight='bold',
            transform=ax.get_xaxis_transform())

fig, axes = plt.subplots(1, 3, figsize=(13, 6), sharey=True)
fig.patch.set_facecolor('white')

panels = [
    ('Gummow', gummow_alone,  None),
    ('Hayne',  hayne_alone,   None),
    ('Gummow & Hayne\ntogether', both_together, None),
]

for ax, (title, tog, alone) in zip(axes, panels):
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
    ax.set_axisbelow(True)

    if alone is not None:
        bp = ax.boxplot(
            [tog, alone],
            positions=[1, 2],
            widths=0.5,
            patch_artist=True,
            medianprops=dict(color=RED, linewidth=2),
            whiskerprops=dict(color='#555'),
            capprops=dict(color='#555'),
            flierprops=dict(marker='o', markersize=3, alpha=0.4,
                            markerfacecolor='#999', markeredgecolor='none'),
        )
        bp['boxes'][0].set_facecolor(DARK)
        bp['boxes'][0].set_alpha(0.75)
        bp['boxes'][1].set_facecolor(LIGHT)
        bp['boxes'][1].set_alpha(0.75)

        ax.set_xticks([1, 2])
        ax.set_xticklabels(['Together', 'Alone'], fontsize=10)
        add_stats(ax, tog,   1, DARK)
        add_stats(ax, alone, 2, '#777')
    else:
        bp = ax.boxplot(
            [tog],
            positions=[1.5],
            widths=0.5,
            patch_artist=True,
            medianprops=dict(color=RED, linewidth=2),
            whiskerprops=dict(color='#555'),
            capprops=dict(color='#555'),
            flierprops=dict(marker='o', markersize=3, alpha=0.4,
                            markerfacecolor='#999', markeredgecolor='none'),
        )
        bp['boxes'][0].set_facecolor(DARK)
        bp['boxes'][0].set_alpha(0.75)
        ax.set_xticks([1.5])
        ax.set_xticklabels(['Together'], fontsize=10)
        add_stats(ax, tog, 1.5, DARK)
        ax.set_xlim(0.8, 2.2)

    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_ylim(-0.05, 1.1)

axes[0].set_ylabel('Relative citation score', fontsize=11)

fig.suptitle('Relative Citation Score — Writing Together vs. Alone',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('images/citation_boxplots.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved: citation_boxplots.png")

Saved: citation_boxplots.png


In [16]:
for case in data.values():
    sorted_j = sorted(case['judgmentAuthors'], key=lambda j: j.get('totalCitations', 0), reverse=True)
    top = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
    for rank, j in enumerate(sorted_j, start=1):
        j['citationRank'] = rank
        j['relativeCitation'] = round(j.get('totalCitations', 0) / top, 4) if top > 0 else 0
    case['judgmentAuthors'] = sorted_j
 
judge_rc       = defaultdict(list)
gh_together_rc = []
 
for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        rc    = j['relativeCitation']
        if 'GUMMOW' in names and 'HAYNE' in names:
            gh_together_rc.append(rc)
        for name in names:
            if name not in ('GUMMOW', 'HAYNE'):
                judge_rc[name].append(rc)
 
JUDGES = sorted(judge_rc.keys())
GH     = '#1a5276'
DARK   = '#333333'
RED    = '#cc3333'
 
all_panels = [('Gummow\n& Hayne', gh_together_rc, GH)] + \
             [(j.capitalize(), judge_rc[j], DARK) for j in JUDGES]
 
n_total = len(all_panels)
n_cols  = (n_total + 1) // 2
fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 12), sharey=True)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()
 
def draw_box(ax, vals, color, title):
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
 
    bp = ax.boxplot(
        [vals], positions=[1], widths=0.5, patch_artist=True,
        medianprops=dict(color=RED, linewidth=2),
        whiskerprops=dict(color='#555'),
        capprops=dict(color='#555'),
        flierprops=dict(marker='o', markersize=2.5, alpha=0.35,
                        markerfacecolor='#999', markeredgecolor='none'),
    )
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.8)
 
    ax.set_xticks([1])
    ax.set_xticklabels([''], fontsize=8.5)
    ax.set_xlim(0.5, 1.5)
    ax.set_ylim(-0.05, 1.1)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.text(1, -0.07, f'n={len(vals)}', ha='center', va='top',
            fontsize=8, color='#777', transform=ax.get_xaxis_transform())
    ax.text(1, -0.13, f'med={np.median(vals):.2f}', ha='center', va='top',
            fontsize=8, color='#333', fontweight='bold',
            transform=ax.get_xaxis_transform())
 
for i, (title, vals, color) in enumerate(all_panels):
    draw_box(axes_flat[i], vals, color, title)
 
# Hide any unused axes
for j in range(len(all_panels), len(axes_flat)):
    axes_flat[j].set_visible(False)
 
# y-axis label only on leftmost of each row
for row in range(2):
    axes[row][0].set_ylabel('Relative citation score', fontsize=11)
 
fig.suptitle('Relative Citation Score: G+H together vs. each other judge',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/citation_all_judges.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [17]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data

TOP99 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

judge_rc       = defaultdict(list)
gh_together_rc = []

for citation in TOP99:
    if citation not in data:
        continue
    case = data[citation]
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        rc    = j.get('totalCitations', 0)
        if 'GUMMOW' in names and 'HAYNE' in names:
            gh_together_rc.append(rc)
        for name in names:
            if name not in ('GUMMOW', 'HAYNE'):
                judge_rc[name].append(rc)

JUDGES = sorted(judge_rc.keys())
GH   = '#1a5276'
DARK = '#333333'
RED  = '#cc3333'

all_panels = [('Gummow\n& Hayne', gh_together_rc, GH)] + \
             [(j.capitalize(), judge_rc[j], DARK) for j in JUDGES]

n_total = len(all_panels)
n_cols  = (n_total + 1) // 2
fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 12), sharey=True)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

def draw_box(ax, vals, color, title):
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

    bp = ax.boxplot(
        [vals], positions=[1], widths=0.5, patch_artist=True,
        medianprops=dict(color=RED, linewidth=2),
        whiskerprops=dict(color='#555'),
        capprops=dict(color='#555'),
        flierprops=dict(marker='o', markersize=2.5, alpha=0.35,
                        markerfacecolor='#999', markeredgecolor='none'),
    )
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.8)

    ax.set_xticks([1])
    ax.set_xticklabels([''], fontsize=8.5)
    ax.set_xlim(0.5, 1.5)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.text(1, -0.04, f'n={len(vals)}', ha='center', va='top',
            fontsize=8, color='#777', transform=ax.get_xaxis_transform())
    ax.text(1, -0.10, f'med={np.median(vals):.0f}', ha='center', va='top',
            fontsize=8, color='#333', fontweight='bold',
            transform=ax.get_xaxis_transform())

for i, (title, vals, color) in enumerate(all_panels):
    draw_box(axes_flat[i], vals, color, title)

for j in range(len(all_panels), len(axes_flat)):
    axes_flat[j].set_visible(False)

for row in range(2):
    axes[row][0].set_ylabel('Raw citations', fontsize=11)

fig.suptitle('Raw Citation Count: G+H together vs. each other judge\n(top 99 most cited cases)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/citation_raw_top99.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [18]:
print(len(backup_cases_all))

781


In [23]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data

TOP99 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[1997] HCA 56','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

def gh_wrote_together(case):
    return any(
        'GUMMOW' in [n.strip() for n in j['name'].split(',')]
        and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
        for j in case['judgmentAuthors']
    )

BAR_COLOR = '#333333'
LIGHT     = '#aaaaaa'
RED       = '#cc3333'

# ══════════════════════════════════════════════════════════
# GRAPH 1 — G+H together per decile
# ══════════════════════════════════════════════════════════
decile_together = []
decile_not      = []
decile_labels   = []

for d in range(10):
    decile = TOP99[d*10:(d+1)*10]
    tog = sum(1 for c in decile if c in data and gh_wrote_together(data[c]))
    decile_together.append(tog)
    decile_not.append(10 - tog)
    decile_labels.append(f'{d*10+1}–{(d+1)*10}')

x = np.arange(len(decile_labels))

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

b1 = ax.bar(x, decile_together, color=BAR_COLOR, edgecolor='white', linewidth=0.6,
            zorder=3, label='Wrote together')
b2 = ax.bar(x, decile_not, bottom=decile_together, color=LIGHT, edgecolor='white',
            linewidth=0.6, zorder=3, label='Did not write together')

for i, (tog, notg) in enumerate(zip(decile_together, decile_not)):
    ax.text(i, tog / 2, str(tog), ha='center', va='center',
            fontsize=10, color='white', fontweight='bold', zorder=4)
    ax.text(i, tog + notg / 2, str(notg), ha='center', va='center',
            fontsize=10, color='#555', fontweight='bold', zorder=4)

ax.set_xticks(x)
ax.set_xticklabels([f'#{l}' for l in decile_labels], fontsize=10)
ax.set_xlabel('Citation rank (top 100 cases)', fontsize=11)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_title('Gummow & Hayne — Co-authorship across citation deciles (top 100 cases)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_ylim(0, 12)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa')

plt.tight_layout()
plt.savefig('images/top99_decile.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: top99_decile.png")

# ══════════════════════════════════════════════════════════
# GRAPH 2 — together vs not by tag (top 99 only)
# ════════════════════════════════════════════════════════
together     = defaultdict(int)
not_together = defaultdict(int)
tag_totals   = defaultdict(int)

for citation in TOP99:
    if citation not in data:
        continue
    case = data[citation]
    tags = case['law_areas']
    gh   = gh_wrote_together(case)
    for tag in tags:
        tag_totals[tag] += 1
        if gh:
            together[tag] += 1
        else:
            not_together[tag] += 1

# Sort by together % descending
tags_sorted = sorted(tag_totals.keys(),
                     key=lambda t: together[t] / tag_totals[t], reverse=True)
tog  = np.array([together[t]     for t in tags_sorted])
notg = np.array([not_together[t] for t in tags_sorted])
tots = tog + notg
labels = [t.replace(' & ', '\n& ').replace(', Planning', ',\nPlanning') for t in tags_sorted]
x = np.arange(len(tags_sorted))

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

ax.bar(x, tog,  color=BAR_COLOR, label='Wrote together',          edgecolor='white', linewidth=0.6, zorder=3)
ax.bar(x, notg, color=LIGHT,     label='Did not write together',   edgecolor='white', linewidth=0.6,
       bottom=tog, zorder=3)

for i, (t, total) in enumerate(zip(tog, tots)):
    if t > 0:
        pct = t / total
        ax.text(i, t / 2, f'{pct:.0%}', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold', zorder=4)
for i, total in enumerate(tots):
    ax.text(i, total + 0.1, f'n={total}', ha='center', va='bottom',
            fontsize=7, color='#555', zorder=4)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8.5)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_title('Gummow & Hayne — Wrote together vs. not, by tag (top 99 cases)\n(sorted by co-authorship rate)',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa')

plt.tight_layout()
plt.savefig('images/top99_tags.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: top99_tags.png")

Saved: top99_decile.png
Saved: top99_tags.png


In [24]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

TOP99 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[1997] HCA 56','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

def gh_wrote_together(case):
    return any(
        'GUMMOW' in [n.strip() for n in j['name'].split(',')]
        and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
        for j in case['judgmentAuthors']
    )

together     = defaultdict(int)
not_together = defaultdict(int)
tag_totals   = defaultdict(int)

for citation in TOP99:
    if citation not in data:
        continue
    case = data[citation]
    tag  = case['law_areas'][0]   # first tag only
    if tag in ('Immigration & Refugees', 'Administrative Law'):
        tag = 'Admin Law & Immigration'
    gh   = gh_wrote_together(case)
    tag_totals[tag] += 1
    if gh:
        together[tag] += 1
    else:
        not_together[tag] += 1

# Sort by together % descending
tags_sorted = sorted(tag_totals.keys(),
                     key=lambda t: together[t] / tag_totals[t], reverse=True)
tog    = np.array([together[t]     for t in tags_sorted])
notg   = np.array([not_together[t] for t in tags_sorted])
tots   = tog + notg
labels = [t.replace(' & ', '\n& ').replace(', Planning', ',\nPlanning') for t in tags_sorted]
x      = np.arange(len(tags_sorted))

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

ax.bar(x, tog,  color='#333333', label='Wrote together',        edgecolor='white', linewidth=0.6, zorder=3)
ax.bar(x, notg, color='#aaaaaa', label='Did not write together', edgecolor='white', linewidth=0.6,
       bottom=tog, zorder=3)

for i, (t, total) in enumerate(zip(tog, tots)):
    if t > 0:
        ax.text(i, t / 2, f'{t/total:.0%}', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold', zorder=4)
for i, total in enumerate(tots):
    ax.text(i, total + 0.1, f'n={total}', ha='center', va='bottom',
            fontsize=7, color='#555', zorder=4)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8.5)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_title('Gummow & Hayne — Wrote together vs. not, by primary tag (top 100 cases)\n'
             '(sorted by co-authorship rate; one tag per case)',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa')

plt.tight_layout()
plt.savefig('images/top99_tags_one_tag.png', dpi=150, bbox_inches='tight', facecolor='white')
print("Saved: top99_tags_one_tag.png")

Saved: top99_tags_one_tag.png


In [25]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

TOP99 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[1997] HCA 56','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

# Add relativeCitation on the fly
for case in data.values():
    sorted_j = sorted(case['judgmentAuthors'], key=lambda j: j.get('totalCitations', 0), reverse=True)
    top = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
    for rank, j in enumerate(sorted_j, start=1):
        j['citationRank'] = rank
        j['relativeCitation'] = round(j.get('totalCitations', 0) / top, 4) if top > 0 else 0
    case['judgmentAuthors'] = sorted_j

judge_rc       = defaultdict(list)
gh_together_rc = []

for citation in TOP99:
    if citation not in data:
        continue
    case = data[citation]
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        rc    = j['relativeCitation']
        if 'GUMMOW' in names and 'HAYNE' in names:
            gh_together_rc.append(rc)
        for name in names:
            if name not in ('GUMMOW', 'HAYNE'):
                judge_rc[name].append(rc)

JUDGES = sorted(judge_rc.keys())
GH   = '#1a5276'
DARK = '#333333'
RED  = '#cc3333'

all_panels = [('Gummow\n& Hayne', gh_together_rc, GH)] + \
             [(j.capitalize(), judge_rc[j], DARK) for j in JUDGES]

n_total = len(all_panels)
n_cols  = (n_total + 1) // 2
fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 12), sharey=True)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

def draw_box(ax, vals, color, title):
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

    bp = ax.boxplot(
        [vals], positions=[1], widths=0.5, patch_artist=True,
        medianprops=dict(color=RED, linewidth=2),
        whiskerprops=dict(color='#555'),
        capprops=dict(color='#555'),
        flierprops=dict(marker='o', markersize=4, alpha=0.9,
                        markerfacecolor='#222', markeredgecolor='#222'),
    )
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.8)

    ax.set_xticks([1])
    ax.set_xticklabels([''], fontsize=8.5)
    ax.set_xlim(0.5, 1.5)
    ax.set_ylim(-0.05, 1.1)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.text(1, -0.07, f'n={len(vals)}', ha='center', va='top',
            fontsize=8, color='#777', transform=ax.get_xaxis_transform())
    ax.text(1, -0.13, f'med={np.median(vals):.2f}', ha='center', va='top',
            fontsize=8, color='#333', fontweight='bold',
            transform=ax.get_xaxis_transform())

for i, (title, vals, color) in enumerate(all_panels):
    draw_box(axes_flat[i], vals, color, title)

for j in range(len(all_panels), len(axes_flat)):
    axes_flat[j].set_visible(False)

for row in range(2):
    axes[row][0].set_ylabel('Relative citation score', fontsize=11)

fig.suptitle('Relative Citation Score: G+H together vs. each other judge\n(top 99 most cited cases)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/citation_relative_top99.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [26]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

yearly = defaultdict(lambda: {'together': 0, 'total': 0})

for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    yearly[year]['total'] += 1
    if any('GUMMOW' in [n.strip() for n in j['name'].split(',')]
           and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
           for j in case['judgmentAuthors']):
        yearly[year]['together'] += 1

years   = sorted(yearly)
tog     = [yearly[y]['together'] for y in years]
totals  = [yearly[y]['total']    for y in years]
rate    = [yearly[y]['together'] / yearly[y]['total'] for y in years]

BAR_COLOR  = '#333333'
LIGHT      = '#aaaaaa'
LINE_COLOR = '#cc3333'

fig, ax1 = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('white')
ax1.set_facecolor('#f8f8f8')

x = np.arange(len(years))
ax1.bar(x, totals, color=LIGHT,     edgecolor='white', linewidth=0.6, zorder=3, label='Total cases')
ax1.bar(x, tog,    color=BAR_COLOR, edgecolor='white', linewidth=0.6, zorder=4, label='G+H wrote together')

ax1.set_xticks(x)
ax1.set_xticklabels(years, fontsize=10)
ax1.set_ylabel('Number of cases', fontsize=11)
ax1.set_xlabel('Year', fontsize=11)
ax1.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax1.set_axisbelow(True)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Rate line on secondary axis
ax2 = ax1.twinx()
ax2.plot(x, rate, color=LINE_COLOR, linewidth=2, marker='o',
         markersize=5, zorder=5, label='Co-authorship rate')
ax2.set_ylabel('Co-authorship rate', fontsize=11, color=LINE_COLOR)
ax2.tick_params(axis='y', labelcolor=LINE_COLOR)
ax2.set_ylim(0, 1)
ax2.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax2.spines['top'].set_visible(False)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa', loc='upper left')

ax1.set_title('Gummow & Hayne — Co-authorship over time', fontsize=13, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('images/gh_over_time.png', dpi=150, bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [27]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data= backup_cases_all

yearly_tags = defaultdict(lambda: defaultdict(int))
for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    if any('GUMMOW' in [n.strip() for n in j['name'].split(',')]
           and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
           for j in case['judgmentAuthors']):
        tag = case['law_areas'][0]
        if tag in ('Administrative Law', 'Immigration & Refugees'):
            tag = 'Admin Law & Immigration'
        yearly_tags[year][tag] += 1

years = sorted(yearly_tags)

# Order tags by total frequency descending (for legend clarity)
tag_totals = defaultdict(int)
for y in yearly_tags.values():
    for tag, count in y.items():
        tag_totals[tag] += count
all_tags = sorted(tag_totals, key=lambda t: -tag_totals[t])

# Assign colors — use tab20 for enough distinct colors
cmap   = plt.cm.tab20
colors = {tag: cmap(i / len(all_tags)) for i, tag in enumerate(all_tags)}

x      = np.arange(len(years))
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

bottoms = np.zeros(len(years))
for tag in all_tags:
    vals = np.array([yearly_tags[y].get(tag, 0) for y in years])
    ax.bar(x, vals, bottom=bottoms, color=colors[tag],
           edgecolor='white', linewidth=0.4, zorder=3, label=tag)
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels(years, fontsize=10)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_xlabel('Year', fontsize=11)
ax.set_title('Gummow & Hayne — Cases written together by tag over time',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=8.5, frameon=True, facecolor='white', edgecolor='#aaa',
          bbox_to_anchor=(1.01, 1), loc='upper left', title='Tag', title_fontsize=9)

plt.tight_layout()
plt.savefig('images/gh_over_time_tags.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [28]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
from collections import defaultdict

data = backup_cases_all

def merge_tag(tag):
    if tag in ('Administrative Law', 'Immigration & Refugees'):
        return 'Admin Law & Immigration'
    return tag

yearly_together = defaultdict(lambda: defaultdict(int))
yearly_total    = defaultdict(lambda: defaultdict(int))

for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    tag  = merge_tag(case['law_areas'][0])
    yearly_total[year][tag] += 1
    if any('GUMMOW' in [n.strip() for n in j['name'].split(',')]
           and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
           for j in case['judgmentAuthors']):
        yearly_together[year][tag] += 1

years    = sorted(yearly_total.keys())
all_tags = sorted({t for y in yearly_total.values() for t in y})
all_tags = [t for t in all_tags
            if sum(yearly_total[y].get(t, 0) for y in years) >= 5]
all_tags = sorted(all_tags)

n_tags = len(all_tags)
n_cols = 4
n_rows = (n_tags + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 2.2),
                         sharey=True, sharex=False)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

BAR_COLOR = '#333333'
LIGHT     = '#dddddd'

for i, tag in enumerate(all_tags):
    ax = axes_flat[i]
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    tag_years = [y for y in years if yearly_total[y].get(tag, 0) > 0]
    tog   = np.array([yearly_together[y].get(tag, 0) for y in tag_years], dtype=float)
    total = np.array([yearly_total[y].get(tag, 0)    for y in tag_years], dtype=float)
    rate  = tog / total  # proportion

    x = np.arange(len(tag_years))
    ax.bar(x, np.ones(len(tag_years)), color=LIGHT,     edgecolor='white', linewidth=0.3, zorder=2)
    ax.bar(x, rate,                    color=BAR_COLOR,  edgecolor='white', linewidth=0.3, zorder=3)

    # n= total cases below each bar
    for xi, (y, n) in enumerate(zip(tag_years, total.astype(int))):
        ax.text(xi, -0.08, str(n), ha='center', va='top', fontsize=5.5,
                color='#888', transform=ax.get_xaxis_transform())

    ax.set_xticks(x)
    ax.set_xticklabels([str(y)[2:] for y in tag_years], fontsize=6.5, rotation=45, ha='right')
    ax.set_title(tag, fontsize=8.5, fontweight='bold', pad=5)
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
    ax.tick_params(axis='y', labelsize=6.5)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

for j in range(len(all_tags), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Gummow & Hayne — Co-authorship rate by tag over time\n'
             '(dark = G+H together as % of total cases in that tag/year; n = total cases)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=1.8, w_pad=1.0)
plt.savefig('images/gh_tag_over_time.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [29]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

# Add relativeCitation on the fly
for case in data.values():
    sorted_j = sorted(case['judgmentAuthors'], key=lambda j: j.get('totalCitations', 0), reverse=True)
    top = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
    for rank, j in enumerate(sorted_j, start=1):
        j['citationRank'] = rank
        j['relativeCitation'] = round(j.get('totalCitations', 0) / top, 4) if top > 0 else 0
    case['judgmentAuthors'] = sorted_j

gummow_all,  hayne_all,  gh_together = [], [], []
gummow_solo, hayne_solo              = [], []

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        rc    = j['relativeCitation']
        has_g = 'GUMMOW' in names
        has_h = 'HAYNE'  in names

        if has_g and has_h:
            gh_together.append(rc)
            gummow_all.append(rc)
            hayne_all.append(rc)
        elif has_g:
            gummow_all.append(rc)
            gummow_solo.append(rc)
        elif has_h:
            hayne_all.append(rc)
            hayne_solo.append(rc)

GH   = '#1a5276'
DARK = '#444444'
RED  = '#cc3333'

def make_fig(panels, title, filename):
    fig, axes = plt.subplots(1, 3, figsize=(13, 6), sharey=True)
    fig.patch.set_facecolor('white')

    for ax, (label, vals, color, xlabel) in zip(axes, panels):
        ax.set_facecolor('#f8f8f8')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
        ax.set_axisbelow(True)

        bp = ax.boxplot(
            [vals], positions=[1], widths=0.5, patch_artist=True,
            medianprops=dict(color=RED, linewidth=2),
            whiskerprops=dict(color='#555'),
            capprops=dict(color='#555'),
            flierprops=dict(marker='o', markersize=4, alpha=0.9,
                            markerfacecolor='#222', markeredgecolor='#222'),
        )
        bp['boxes'][0].set_facecolor(color)
        bp['boxes'][0].set_alpha(0.8)

        ax.set_xticks([1])
        ax.set_xticklabels([xlabel], fontsize=10)
        ax.set_xlim(0.5, 1.5)
        ax.set_ylim(-0.05, 1.1)
        ax.set_title(label, fontsize=12, fontweight='bold', pad=10)
        ax.text(1, -0.07, f'n={len(vals)}', ha='center', va='top',
                fontsize=9, color='#777', transform=ax.get_xaxis_transform())
        ax.text(1, -0.13, f'med={np.median(vals):.2f}', ha='center', va='top',
                fontsize=9, color='#333', fontweight='bold',
                transform=ax.get_xaxis_transform())

    axes[0].set_ylabel('Relative citation score', fontsize=11)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

# Graph 1 — all judgments (including joint)
make_fig(
    panels=[
        ('Gummow',          gummow_all,  DARK, 'All judgments'),
        ('Hayne',           hayne_all,   DARK, 'All judgments'),
        ('Gummow & Hayne\ntogether', gh_together, GH, 'Together'),
    ],
    title='Relative Citation Score — All judgments (incl. joint)',
    filename='images/citation_all_judgments.png'
)

# Graph 2 — solo judgments only
make_fig(
    panels=[
        ('Gummow',          gummow_solo, DARK, 'Solo only'),
        ('Hayne',           hayne_solo,  DARK, 'Solo only'),
        ('Gummow & Hayne\ntogether', gh_together, GH, 'Together'),
    ],
    title='Relative Citation Score — Solo judgments vs. G+H together',
    filename='images/citation_solo_judgments.png'
)

Saved: images/citation_all_judgments.png
Saved: images/citation_solo_judgments.png


In [ ]:
# FIX
data = backup_cases_all

rows = []
for citation, case in data.items():
    bench = ', '.join(case.get('bench', []))
    for p in case['paragraphs']:
        rows.append({
            'rank':          None,
            'para_number':   p['para'],
            'case':          citation,
            'judgment_authors': p['judge'],
            'bench':         bench,
            'citations':     p.get('citations', 0),
            'snippet':       p.get('snippet', ''),
        })

# Sort by citations descending, take top 1000
rows.sort(key=lambda x: -x['citations'])
rows = rows[:1000]
for i, r in enumerate(rows, start=1):
    r['rank'] = i

# Print to console
print(f"{'Rank':<5} {'Para':>5} {'Citations':>10}  {'Case':<20} {'Judgment authors':<50} {'Bench'}")
print("-" * 160)
for r in rows:
    print(f"{r['rank']:<5} {r['para_number']:>5} {r['citations']:>10,}  {r['case']:<20} {r['judgment_authors']:<50} {r['bench']}")

"""# Also save as CSV
with open('top1000_paragraphs.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['rank','para_number','case','judgment_authors','bench','citations','snippet'])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nSaved: top1000_paragraphs.csv ({len(rows)} rows)")"""

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
import numpy as np
 
data = backup_cases_all
 
majority_count = defaultdict(int)
total_count    = defaultdict(int)
 
for case in data.values():
    bench_size = len(case.get('bench', []))
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        is_majority = len(names) > bench_size / 2 if bench_size > 0 else False
        for name in names:
            total_count[name] += 1
            if is_majority:
                majority_count[name] += 1
 
judges  = sorted(total_count.keys())
totals  = [total_count[j]    for j in judges]
majors  = [majority_count[j] for j in judges]
rates   = [majority_count[j] / total_count[j] for j in judges]
 
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')
 
from adjustText import adjust_text
 
ax.scatter(totals, majors, s=80, color='#333333', zorder=3)
 
texts = []
for judge, x, y, rate in zip(judges, totals, majors, rates):
    texts.append(ax.text(x, y, f"{judge.capitalize()} ({rate:.0%})",
                         fontsize=8.5, color='#222'))
 
adjust_text(texts, x=totals, y=majors, ax=ax,
            expand=(1.0, 1.0))
 
# Reference line: y = 0.5x (50% majority rate)
x_line = np.linspace(0, max(totals) * 1.05, 100)
ax.plot(x_line, x_line * 0.5, color='#cc3333', linewidth=1,
        linestyle='--', alpha=0.6, label='50% majority rate', zorder=2)
 
ax.set_xlabel('Total judgments written', fontsize=12)
ax.set_ylabel('Judgments in majority', fontsize=12)
ax.set_title('Judge majority judgment rate vs total output', fontsize=14, fontweight='bold', pad=14)
ax.legend(fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa')
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.xaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
 
plt.tight_layout()
plt.savefig('images/majority_scatter.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [10]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from itertools import combinations

data = backup_cases_all

comat        = defaultdict(int)
judge_totals = defaultdict(int)

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            judge_totals[name] += 1
        for a, b in combinations(sorted(names), 2):
            comat[(a, b)] += 1

# Sort judges by total judgments descending
JUDGES = sorted(judge_totals.keys(), key=lambda j: -judge_totals[j])
n      = len(JUDGES)
idx    = {j: i for i, j in enumerate(JUDGES)}

# Build matrix — off-diagonal = co-authored count, diagonal = total judgments
matrix = np.zeros((n, n), dtype=int)
for i, j in enumerate(JUDGES):
    matrix[i, i] = judge_totals[j]
for (a, b), count in comat.items():
    i, j = idx[a], idx[b]
    matrix[i, j] = count
    matrix[j, i] = count

# Mask diagonal for colorscale (show separately)
off_diag = matrix.copy()
np.fill_diagonal(off_diag, 0)

labels = [j.capitalize() for j in JUDGES]

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

im = ax.imshow(off_diag, cmap='YlOrRd', aspect='auto',
               vmin=0, vmax=off_diag.max())

# Annotate all cells
for i in range(n):
    for j in range(n):
        val = matrix[i, j]
        if i == j:
            # Diagonal — grey background, total judgments
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                        facecolor='#d0d0d0', zorder=1))
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=8, color='#333', fontweight='bold', zorder=2)
        elif val > 0:
            text_color = 'white' if off_diag[i, j] > off_diag.max() * 0.6 else '#333'
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=8, color=text_color, zorder=2)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=9.5)
ax.set_yticklabels(labels, fontsize=9.5)

cbar = plt.colorbar(im, ax=ax, shrink=0.75)
cbar.set_label('Co-authored judgments', fontsize=10)

ax.set_title('Judge co-authorship heatmap\n(diagonal = total judgments written; off-diagonal = joint judgments)',
             fontsize=13, fontweight='bold', pad=14)

plt.tight_layout()
plt.savefig('images/collab_heatmap.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [12]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from itertools import combinations

data = backup_cases_all

comat        = defaultdict(int)
judge_totals = defaultdict(int)

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            judge_totals[name] += 1
        for a, b in combinations(sorted(names), 2):
            comat[(a, b)] += 1

# Sort judges by total judgments descending
JUDGES = sorted(judge_totals.keys(), key=lambda j: -judge_totals[j])
n      = len(JUDGES)
idx    = {j: i for i, j in enumerate(JUDGES)}

# Build matrix — off-diagonal = co-authored count, diagonal = total judgments
matrix = np.zeros((n, n), dtype=int)
for i, j in enumerate(JUDGES):
    matrix[i, i] = judge_totals[j]
for (a, b), count in comat.items():
    i, j = idx[a], idx[b]
    matrix[i, j] = count
    matrix[j, i] = count

# Mask diagonal for colorscale (show separately)
off_diag = matrix.copy()
np.fill_diagonal(off_diag, 0)

labels = [j.capitalize() for j in JUDGES]

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

im = ax.imshow(off_diag, cmap='YlOrRd', aspect='auto',
               vmin=0, vmax=off_diag.max())

# Annotate all cells
for i in range(n):
    for j in range(n):
        val = matrix[i, j]
        if i == j:
            # Diagonal — grey background, total judgments
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                        facecolor='#d0d0d0', zorder=1))
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=8, color='#333', fontweight='bold', zorder=2)
        elif val > 0:
            text_color = 'white' if off_diag[i, j] > off_diag.max() * 0.6 else '#333'
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=8, color=text_color, zorder=2)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=9.5)
ax.set_yticklabels(labels, fontsize=9.5)

cbar = plt.colorbar(im, ax=ax, shrink=0.75)
cbar.set_label('Co-authored judgments', fontsize=10)

ax.set_title('Judge co-authorship heatmap\n(diagonal = total judgments written; off-diagonal = joint judgments)',
             fontsize=13, fontweight='bold', pad=14)

# Build normalised matrix — 2C/(A+B)
norm_matrix = np.zeros((n, n))
for i, ji in enumerate(JUDGES):
    norm_matrix[i, i] = 1.0  # diagonal = 1 by definition
    for j, jj in enumerate(JUDGES):
        if i != j:
            a = judge_totals[ji]
            b = judge_totals[jj]
            pair = tuple(sorted([ji, jj]))
            c    = comat.get(pair, 0)
            norm_matrix[i, j] = (2 * c) / (a + b) if (a + b) > 0 else 0

off_diag_norm = norm_matrix.copy()
np.fill_diagonal(off_diag_norm, 0)

fig2, ax2 = plt.subplots(figsize=(12, 10))
fig2.patch.set_facecolor('white')
ax2.set_facecolor('white')

im2 = ax2.imshow(off_diag_norm, cmap='YlOrRd', aspect='auto',
                 vmin=0, vmax=off_diag_norm.max())

for i in range(n):
    for j in range(n):
        if i == j:
            ax2.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                                         facecolor='#d0d0d0', zorder=1))
            ax2.text(j, i, '—', ha='center', va='center',
                     fontsize=9, color='#555', zorder=2)
        elif norm_matrix[i, j] > 0:
            text_color = 'white' if off_diag_norm[i, j] > off_diag_norm.max() * 0.6 else '#333'
            ax2.text(j, i, f'{norm_matrix[i, j]:.2f}', ha='center', va='center',
                     fontsize=7.5, color=text_color, zorder=2)

ax2.set_xticks(range(n))
ax2.set_yticks(range(n))
ax2.set_xticklabels(labels, rotation=40, ha='right', fontsize=9.5)
ax2.set_yticklabels(labels, fontsize=9.5)

cbar2 = plt.colorbar(im2, ax=ax2, shrink=0.75)
cbar2.set_label('Normalised co-authorship score', fontsize=10)

ax2.set_title('Judge co-authorship heatmap (normalised)',
              fontsize=13, fontweight='bold', pad=14)

plt.tight_layout()
plt.savefig('images/collab_heatmap_normalised.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: collab_heatmap_normalised.png")

Saved: collab_heatmap_normalised.png


In [14]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict, Counter
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

data = backup_cases_all

judge_sizes     = defaultdict(list)
gh_collab_sizes = defaultdict(list)
h_collab_sizes  = defaultdict(list)  # GUMMOW wrote with HAYNE
g_collab_sizes  = defaultdict(list)  # HAYNE wrote with GUMMOW

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        has_g = 'GUMMOW' in names
        has_h = 'HAYNE'  in names
        for name in names:
            judge_sizes[name].append(len(names))
        for name in names:
            if name not in ('GUMMOW', 'HAYNE') and has_g and has_h:
                gh_collab_sizes[name].append(len(names))
        if has_g and has_h:
            h_collab_sizes['GUMMOW'].append(len(names))
            g_collab_sizes['HAYNE'].append(len(names))

collab_dots = {**gh_collab_sizes,
               'GUMMOW': h_collab_sizes['GUMMOW'],
               'HAYNE':  g_collab_sizes['HAYNE']}

from matplotlib.colors import LinearSegmentedColormap
green_purple = LinearSegmentedColormap.from_list(
    'green_purple', ['#1a9641', '#74c6c2', '#2166ac', '#4d004b'], N=256)

# Compute global max density for standardised colorscale
global_max = 0
for vals in collab_dots.values():
    if vals:
        counts = Counter(vals)
        global_max = max(global_max, max(counts.values()))

JUDGES  = sorted(judge_sizes.keys(), key=lambda j: -np.median(judge_sizes[j]))
n_total = len(JUDGES)
n_cols  = (n_total + 1) // 2

np.random.seed(42)

fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 13), sharey=True)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

VIOLIN_C = '#c4a882'

for i, judge in enumerate(JUDGES):
    ax   = axes_flat[i]
    vals = judge_sizes[judge]
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

    vp = ax.violinplot([vals], positions=[1], widths=0.7,
                       showmedians=True, showextrema=True)
    vp['bodies'][0].set_facecolor(VIOLIN_C)
    vp['bodies'][0].set_alpha(0.45)
    vp['cmedians'].set_color('#ff0000')
    vp['cmedians'].set_linewidth(3.5)
    for part in ('cmins', 'cmaxes', 'cbars'):
        vp[part].set_color('#888')

    gh_vals = collab_dots.get(judge, [])
    if gh_vals:
        counts     = Counter(gh_vals)
        jitter     = np.random.uniform(-0.15, 0.15, size=len(gh_vals))
        cmap       = green_purple
        dot_colors = [cmap(counts[v] / global_max) for v in gh_vals]
        for xi, yi, ci in zip(1 + jitter, gh_vals, dot_colors):
            ax.scatter(xi, yi, s=14, color=ci, alpha=0.85, zorder=4)

    gh_n = len(gh_vals)
    ax.set_xticks([1])
    ax.set_xticklabels([''])
    ax.set_xlim(0.5, 1.5)
    ax.set_title(judge.capitalize(), fontsize=10, fontweight='bold', pad=8)

    # Three lines below: n=, med=, G+H count — with more spacing
    ax.text(1, -0.05, f'n={len(vals)}',     ha='center', va='top', fontsize=7.5,
            color='#777', transform=ax.get_xaxis_transform())
    ax.text(1, -0.12, f'med={np.median(vals):.1f}', ha='center', va='top', fontsize=7.5,
            color='#333', fontweight='bold', transform=ax.get_xaxis_transform())
    ax.text(1, -0.19, f'G+H: {gh_n}',       ha='center', va='top', fontsize=7.5,
            color='#1a5276', fontweight='bold', transform=ax.get_xaxis_transform())

for j in range(len(JUDGES), len(axes_flat)):
    axes_flat[j].set_visible(False)

for row in range(2):
    axes[row][0].set_ylabel('Number of judges in judgment', fontsize=11)

for ax in axes_flat[:len(JUDGES)]:
    ax.set_yticks(range(1, 8))
    ax.set_ylim(0.5, 7.5)

# Colorbar + median legend at bottom
fig.subplots_adjust(bottom=0.14, top=0.93, hspace=0.45)
cbar_ax = fig.add_axes([0.25, 0.05, 0.4, 0.015])
sm = ScalarMappable(cmap=green_purple, norm=Normalize(vmin=0, vmax=global_max))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_label('G+H dot density', fontsize=9)
mid = global_max // 2
cbar.set_ticks([1, mid, global_max])
cbar.set_ticklabels([f'Low (n=1)', f'Medium (n={mid})', f'High (n={global_max})'])

# Median line legend below colorbar
median_line = Line2D([0], [0], color='#ff0000', linewidth=3.5, label='Median')
fig.legend(handles=[median_line], loc='lower center',
           bbox_to_anchor=(0.5, 0.0), fontsize=9,
           frameon=True, facecolor='white', edgecolor='#aaa')

fig.suptitle('Judgment collaboration size per judge\n(1 = wrote alone, 7 = full bench)',
             fontsize=14, fontweight='bold')
plt.savefig('images/collab_size_violinplot.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [16]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from itertools import combinations

data = backup_cases_all

YEARS = list(range(1998, 2013))

yearly_comat  = defaultdict(lambda: defaultdict(int))
yearly_totals = defaultdict(lambda: defaultdict(int))
yearly_active = defaultdict(set)

for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    if year not in YEARS:
        continue
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            yearly_totals[year][name] += 1
            yearly_active[year].add(name)
        for a, b in combinations(sorted(names), 2):
            yearly_comat[year][(a, b)] += 1

# Global vmax for consistent colorscale
all_raw, all_norm = [], []
for year in YEARS:
    for (a, b), c in yearly_comat[year].items():
        all_raw.append(c)
        ta = yearly_totals[year].get(a, 0)
        tb = yearly_totals[year].get(b, 0)
        if ta + tb > 0:
            all_norm.append((2 * c) / (ta + tb))
raw_vmax  = max(all_raw)  if all_raw  else 1
norm_vmax = max(all_norm) if all_norm else 1

def draw_heatmap(ax, matrix, labels, vmin, vmax, cmap, fmt=None):
    n = len(labels)
    off_diag = matrix.copy()
    np.fill_diagonal(off_diag, 0)
    im = ax.imshow(off_diag, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    for i in range(n):
        for j in range(n):
            val = matrix[i, j]
            if i == j:
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                                            facecolor='#bbbbbb', zorder=1))
                if val > 0:
                    txt = str(int(val)) if fmt is None else f'{val:.2f}'
                    ax.text(j, i, txt, ha='center', va='center',
                            fontsize=7, color='#333', fontweight='bold', zorder=2)
            elif val > 0:
                txt = str(int(val)) if fmt is None else f'{val:.2f}'
                tc  = 'white' if off_diag[i, j] > vmax * 0.6 else '#333'
                ax.text(j, i, txt, ha='center', va='center',
                        fontsize=7, color=tc, zorder=2)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    return im

for year in YEARS:
    active      = yearly_active[year]
    year_judges = sorted(active, key=lambda j: -yearly_totals[year].get(j, 0))
    n_y         = len(year_judges)
    idx_y       = {j: i for i, j in enumerate(year_judges)}
    labels_y    = [j.capitalize() for j in year_judges]

    raw_mat  = np.zeros((n_y, n_y))
    norm_mat = np.zeros((n_y, n_y))

    for i, ji in enumerate(year_judges):
        raw_mat[i, i]  = yearly_totals[year].get(ji, 0)
        norm_mat[i, i] = yearly_totals[year].get(ji, 0)

    for (a, b), c in yearly_comat[year].items():
        if a in idx_y and b in idx_y:
            ia, ib = idx_y[a], idx_y[b]
            raw_mat[ia, ib] = c
            raw_mat[ib, ia] = c
            ta  = yearly_totals[year].get(a, 0)
            tb  = yearly_totals[year].get(b, 0)
            val = (2 * c) / (ta + tb) if (ta + tb) > 0 else 0
            norm_mat[ia, ib] = val
            norm_mat[ib, ia] = val

    cell = max(0.7, 8 / n_y)
    fig, axes = plt.subplots(1, 2, figsize=(n_y * cell * 2 + 4, n_y * cell + 2))
    fig.patch.set_facecolor('white')
    fig.suptitle(f'Judge co-authorship — {year}  ({n_y} active judges)',
                 fontsize=13, fontweight='bold')

    im1 = draw_heatmap(axes[0], raw_mat,  labels_y, 0, raw_vmax,  'YlOrRd')
    axes[0].set_title('Raw co-authored judgments', fontsize=11, fontweight='bold', pad=8)
    plt.colorbar(im1, ax=axes[0], shrink=0.85, label='Co-authored judgments')

    im2 = draw_heatmap(axes[1], norm_mat, labels_y, 0, norm_vmax, 'YlOrRd', fmt='norm')
    axes[1].set_title('Normalised (2C / A+B)', fontsize=11, fontweight='bold', pad=8)
    plt.colorbar(im2, ax=axes[1], shrink=0.85, label='Normalised score')

    plt.tight_layout()
    plt.savefig(f'images/heatmap_{year}.png', dpi=130,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: heatmap_{year}.png")

Saved: heatmap_1998.png
Saved: heatmap_1999.png
Saved: heatmap_2000.png
Saved: heatmap_2001.png
Saved: heatmap_2002.png
Saved: heatmap_2003.png
Saved: heatmap_2004.png
Saved: heatmap_2005.png
Saved: heatmap_2006.png
Saved: heatmap_2007.png
Saved: heatmap_2008.png
Saved: heatmap_2009.png
Saved: heatmap_2010.png
Saved: heatmap_2011.png
Saved: heatmap_2012.png


In [15]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from itertools import combinations

data = backup_cases_all

YEARS = list(range(1998, 2013))
ALL_JUDGES = ['GUMMOW', 'HAYNE', 'GLEESON', 'CRENNAN', 'HEYDON',
              'FRENCH', 'KIEFEL', 'KIRBY', 'McHUGH', 'BELL',
              'CALLINAN', 'GAUDRON', 'BRENNAN', 'TOOHEY']

def make_heatmap(ax, matrix, labels, active, vmin, vmax, cmap, show_diag=True, fmt=None):
    """Draw a heatmap on ax, greying out inactive judges."""
    n = len(labels)
    off_diag = matrix.copy()
    np.fill_diagonal(off_diag, 0)
    im = ax.imshow(off_diag, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)

    for i in range(n):
        for j in range(n):
            val = matrix[i, j]
            judge_i = labels[i].upper() if labels[i] != 'Mchugh' else 'McHUGH'
            judge_j = labels[j].upper() if labels[j] != 'Mchugh' else 'McHUGH'
            inactive = judge_i not in active or judge_j not in active

            if inactive:
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                                            facecolor='#e0e0e0', zorder=1))
                continue

            if i == j:
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                                            facecolor='#bbbbbb', zorder=1))
                if val > 0:
                    ax.text(j, i, str(int(val)) if fmt is None else f'{val:.2f}',
                            ha='center', va='center', fontsize=5.5,
                            color='#333', fontweight='bold', zorder=2)
            elif val > 0:
                text_color = 'white' if off_diag[i,j] > vmax * 0.6 else '#333'
                ax.text(j, i, str(int(val)) if fmt is None else f'{val:.2f}',
                        ha='center', va='center', fontsize=5.5,
                        color=text_color, zorder=2)

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=6)
    ax.set_yticklabels(labels, fontsize=6)
    return im

# Build per-year matrices
yearly_comat  = defaultdict(lambda: defaultdict(int))
yearly_totals = defaultdict(lambda: defaultdict(int))
yearly_active = defaultdict(set)

for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[','').strip())
    if year not in YEARS:
        continue
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            yearly_totals[year][name] += 1
            yearly_active[year].add(name)
        for a, b in combinations(sorted(names), 2):
            yearly_comat[year][(a, b)] += 1

labels = [j.capitalize() for j in ALL_JUDGES]
n      = len(ALL_JUDGES)
idx    = {j: i for i, j in enumerate(ALL_JUDGES)}

# Global vmax for raw and norm across all years (for consistent scale)
all_raw_vals, all_norm_vals = [], []
for year in YEARS:
    for (a, b), c in yearly_comat[year].items():
        all_raw_vals.append(c)
        ta = yearly_totals[year].get(a, 0)
        tb = yearly_totals[year].get(b, 0)
        if ta + tb > 0:
            all_norm_vals.append((2 * c) / (ta + tb))

raw_vmax  = max(all_raw_vals)  if all_raw_vals  else 1
norm_vmax = max(all_norm_vals) if all_norm_vals else 1

# One figure per year, each with 2 subplots (raw + norm)
for year in YEARS:
    active = yearly_active[year]

    # Build raw matrix
    raw_mat = np.zeros((n, n))
    for i, ji in enumerate(ALL_JUDGES):
        raw_mat[i, i] = yearly_totals[year].get(ji, 0)
    for (a, b), c in yearly_comat[year].items():
        if a in idx and b in idx:
            raw_mat[idx[a], idx[b]] = c
            raw_mat[idx[b], idx[a]] = c

    # Build norm matrix
    norm_mat = np.zeros((n, n))
    for i, ji in enumerate(ALL_JUDGES):
        norm_mat[i, i] = yearly_totals[year].get(ji, 0)  # diag = total for context
    for (a, b), c in yearly_comat[year].items():
        if a in idx and b in idx:
            ta = yearly_totals[year].get(a, 0)
            tb = yearly_totals[year].get(b, 0)
            val = (2 * c) / (ta + tb) if (ta + tb) > 0 else 0
            norm_mat[idx[a], idx[b]] = val
            norm_mat[idx[b], idx[a]] = val

    fig, axes = plt.subplots(1, 2, figsize=(20, 9))
    fig.patch.set_facecolor('white')
    fig.suptitle(f'Judge co-authorship heatmap — {year}', fontsize=14, fontweight='bold')

    im1 = make_heatmap(axes[0], raw_mat,  labels, active, 0, raw_vmax,  'YlOrRd')
    axes[0].set_title('Raw co-authored judgments', fontsize=11, fontweight='bold', pad=8)
    plt.colorbar(im1, ax=axes[0], shrink=0.8, label='Co-authored judgments')

    im2 = make_heatmap(axes[1], norm_mat, labels, active, 0, norm_vmax, 'YlOrRd', fmt='norm')
    axes[1].set_title('Normalised (2C / A+B)', fontsize=11, fontweight='bold', pad=8)
    plt.colorbar(im2, ax=axes[1], shrink=0.8, label='Normalised score')

    # Grey = inactive legend note
    axes[0].text(0.01, -0.12, '■ Grey = judge not active that year',
                 transform=axes[0].transAxes, fontsize=8, color='#777')

    plt.tight_layout()
    plt.savefig(f'images/heatmap_{year}.png', dpi=130,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: heatmap_{year}.png")

Saved: heatmap_1998.png
Saved: heatmap_1999.png
Saved: heatmap_2000.png
Saved: heatmap_2001.png
Saved: heatmap_2002.png
Saved: heatmap_2003.png
Saved: heatmap_2004.png
Saved: heatmap_2005.png
Saved: heatmap_2006.png
Saved: heatmap_2007.png
Saved: heatmap_2008.png
Saved: heatmap_2009.png
Saved: heatmap_2010.png
Saved: heatmap_2011.png
Saved: heatmap_2012.png


In [17]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

def merge_tag(tag):
    if tag in ('Administrative Law', 'Immigration & Refugees'):
        return 'Admin Law\n& Immigration'
    return tag

together     = defaultdict(int)
not_together = defaultdict(int)
tag_totals   = defaultdict(int)

for case in data.values():
    tag = merge_tag(case['law_areas'][0])
    gh  = any('GUMMOW' in [n.strip() for n in j['name'].split(',')]
              and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
              for j in case['judgmentAuthors'])
    tag_totals[tag] += 1
    if gh:
        together[tag] += 1
    else:
        not_together[tag] += 1

# Sort by co-authorship rate descending
tags_sorted = sorted(tag_totals.keys(),
                     key=lambda t: together[t] / tag_totals[t], reverse=True)
tog  = np.array([together[t]     for t in tags_sorted])
notg = np.array([not_together[t] for t in tags_sorted])
tots = tog + notg
labels = [t.replace(' & ', '\n& ').replace(', Planning', ',\nPlanning')
            .replace('& Immigration', '& Immigration') for t in tags_sorted]
x = np.arange(len(tags_sorted))

fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

ax.bar(x, tog,  color='#222222', label='Wrote together',          edgecolor='white', linewidth=0.6, zorder=3)
ax.bar(x, notg, color='#aaaaaa', label='Did not write together',  edgecolor='white', linewidth=0.6,
       bottom=tog, zorder=3)

# % inside dark bar
for i, (t, total) in enumerate(zip(tog, tots)):
    if t > 0:
        pct = t / total
        ax.text(i, t / 2, f'{pct:.0%}', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold', zorder=4)

# n= above each bar
for i, total in enumerate(tots):
    ax.text(i, total + 0.3, f'n={total}', ha='center', va='bottom',
            fontsize=7, color='#555', zorder=4)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8.5)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_title('Gummow & Hayne — Wrote together vs. not, by primary tag (all 781 cases)\n'
             '(sorted by co-authorship rate; one tag per case)',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa')

plt.tight_layout()
plt.savefig('images/gh_all_cases_tags.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [18]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

def merge_tag(tag):
    if tag in ('Administrative Law', 'Immigration & Refugees'):
        return 'Admin Law\n& Immigration'
    return tag

together     = defaultdict(int)
not_together = defaultdict(int)
tag_totals   = defaultdict(int)

for case in data.values():
    bench = case.get('bench', [])
    # Only include cases where both G and H were on the bench
    if 'GUMMOW' not in bench or 'HAYNE' not in bench:
        continue

    tag = merge_tag(case['law_areas'][0])
    gh  = any('GUMMOW' in [n.strip() for n in j['name'].split(',')]
              and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
              for j in case['judgmentAuthors'])
    tag_totals[tag] += 1
    if gh:
        together[tag] += 1
    else:
        not_together[tag] += 1

# Sort by co-authorship rate descending
tags_sorted = sorted(tag_totals.keys(),
                     key=lambda t: together[t] / tag_totals[t], reverse=True)
tog    = np.array([together[t]     for t in tags_sorted])
notg   = np.array([not_together[t] for t in tags_sorted])
tots   = tog + notg
labels = [t.replace(' & ', '\n& ').replace(', Planning', ',\nPlanning')
          for t in tags_sorted]
x = np.arange(len(tags_sorted))

fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

ax.bar(x, tog,  color='#222222', label='Wrote together',         edgecolor='white', linewidth=0.6, zorder=3)
ax.bar(x, notg, color='#aaaaaa', label='Did not write together', edgecolor='white', linewidth=0.6,
       bottom=tog, zorder=3)

for i, (t, total) in enumerate(zip(tog, tots)):
    if t > 0:
        ax.text(i, t / 2, f'{t/total:.0%}', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold', zorder=4)

for i, total in enumerate(tots):
    ax.text(i, total + 0.3, f'n={total}', ha='center', va='bottom',
            fontsize=7, color='#555', zorder=4)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8.5)
ax.set_ylabel('Number of cases', fontsize=11)
ax.set_title('Gummow & Hayne — Wrote together vs. not, by primary tag\n'
             '(only cases where both were on the bench; sorted by co-authorship rate)',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa')

plt.tight_layout()
plt.savefig('images/gh_bench_tags.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print(f"Saved. Total cases: {sum(tots)}")

Saved. Total cases: 550


In [19]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

for case in data.values():
    sorted_j = sorted(case['judgmentAuthors'], key=lambda j: j.get('totalCitations', 0), reverse=True)
    top = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
    for rank, j in enumerate(sorted_j, start=1):
        j['citationRank'] = rank
        j['relativeCitation'] = round(j.get('totalCitations', 0) / top, 4) if top > 0 else 0
    case['judgmentAuthors'] = sorted_j

judge_rc       = defaultdict(list)
gh_together_rc = []
gummow_rc      = []
hayne_rc       = []

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        rc    = j['relativeCitation']
        if 'GUMMOW' in names and 'HAYNE' in names:
            gh_together_rc.append(rc)
        if 'GUMMOW' in names:
            gummow_rc.append(rc)
        if 'HAYNE' in names:
            hayne_rc.append(rc)
        for name in names:
            if name not in ('GUMMOW', 'HAYNE'):
                judge_rc[name].append(rc)

JUDGES = sorted(judge_rc.keys())
GH     = '#1a5276'
GBLUE  = '#2980b9'
DARK   = '#333333'
RED    = '#cc3333'

all_panels = [
    ('Gummow\n& Hayne', gh_together_rc, GH),
    ('Gummow',          gummow_rc,       GBLUE),
    ('Hayne',           hayne_rc,        GBLUE),
] + [(j.capitalize(), judge_rc[j], DARK) for j in JUDGES]

n_total = len(all_panels)
n_cols  = (n_total + 1) // 2
fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 12), sharey=True)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

def draw_box(ax, vals, color, title):
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

    bp = ax.boxplot(
        [vals], positions=[1], widths=0.5, patch_artist=True,
        medianprops=dict(color=RED, linewidth=2),
        whiskerprops=dict(color='#555'),
        capprops=dict(color='#555'),
        flierprops=dict(marker='o', markersize=2.5, alpha=0.35,
                        markerfacecolor='#999', markeredgecolor='none'),
    )
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.8)

    ax.set_xticks([1])
    ax.set_xticklabels([''], fontsize=8.5)
    ax.set_xlim(0.5, 1.5)
    ax.set_ylim(-0.05, 1.1)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.text(1, -0.07, f'n={len(vals)}', ha='center', va='top',
            fontsize=8, color='#777', transform=ax.get_xaxis_transform())
    ax.text(1, -0.13, f'med={np.median(vals):.2f}', ha='center', va='top',
            fontsize=8, color='#333', fontweight='bold',
            transform=ax.get_xaxis_transform())

for i, (title, vals, color) in enumerate(all_panels):
    draw_box(axes_flat[i], vals, color, title)

# Hide any unused axes
for j in range(len(all_panels), len(axes_flat)):
    axes_flat[j].set_visible(False)

# y-axis label only on leftmost of each row
for row in range(2):
    axes[row][0].set_ylabel('Relative citation score', fontsize=11)

fig.suptitle('Relative Citation Score: G+H together, Gummow, Hayne vs. each other judge',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/citation_all_judges.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [20]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

for case in data.values():
    sorted_j = sorted(case['judgmentAuthors'], key=lambda j: j.get('totalCitations', 0), reverse=True)
    top = sorted_j[0].get('totalCitations', 0) if sorted_j else 0
    for rank, j in enumerate(sorted_j, start=1):
        j['citationRank'] = rank
        j['relativeCitation'] = round(j.get('totalCitations', 0) / top, 4) if top > 0 else 0
    case['judgmentAuthors'] = sorted_j

judge_rc       = defaultdict(list)
judge_gh_rc    = defaultdict(list)   # rc values when judge wrote with both G+H
gh_together_rc = []
gummow_rc      = []
hayne_rc       = []
gummow_gh_rc   = []   # Gummow's judgments with Hayne
hayne_gh_rc    = []   # Hayne's judgments with Gummow

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        rc    = j['relativeCitation']
        has_g = 'GUMMOW' in names
        has_h = 'HAYNE'  in names
        if has_g and has_h:
            gh_together_rc.append(rc)
            gummow_gh_rc.append(rc)
            hayne_gh_rc.append(rc)
        if 'GUMMOW' in names:
            gummow_rc.append(rc)
        if 'HAYNE' in names:
            hayne_rc.append(rc)
        for name in names:
            if name not in ('GUMMOW', 'HAYNE'):
                judge_rc[name].append(rc)
                if has_g and has_h:
                    judge_gh_rc[name].append(rc)

JUDGES = sorted(judge_rc.keys())
GH     = '#1a5276'
GBLUE  = '#2980b9'
DARK   = '#333333'
RED    = '#cc3333'
VIOLIN_C = '#c4a882'

all_panels = [
    ('Gummow\n& Hayne', gh_together_rc, GH,    gh_together_rc, GH),
    ('Gummow',           gummow_rc,      GBLUE, gummow_gh_rc,   GBLUE),
    ('Hayne',            hayne_rc,       GBLUE, hayne_gh_rc,    GBLUE),
] + [(j.capitalize(), judge_rc[j], DARK, judge_gh_rc[j], None) for j in JUDGES]

n_total = len(all_panels)
n_cols  = (n_total + 1) // 2
fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 12), sharey=True)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

np.random.seed(42)

def draw_box(ax, vals, color, title, gh_dots=None, violin_color=None):
    vc = violin_color if violin_color else VIOLIN_C
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

    vp = ax.violinplot([vals], positions=[1], widths=0.6,
                       showmedians=True, showextrema=True)
    vp['bodies'][0].set_facecolor(vc)
    vp['bodies'][0].set_alpha(0.7)
    vp['cmedians'].set_color(RED)
    vp['cmedians'].set_linewidth(2.5)
    for part in ('cmins', 'cmaxes', 'cbars'):
        vp[part].set_color('#888')

    if gh_dots:
        jitter = np.random.uniform(-0.12, 0.12, size=len(gh_dots))
        ax.scatter(1 + jitter, gh_dots, s=8, color='#1a5276',
                   alpha=0.5, zorder=4)

    ax.set_xticks([1])
    ax.set_xticklabels([''], fontsize=8.5)
    ax.set_xlim(0.5, 1.5)
    ax.set_ylim(-0.05, 1.1)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.text(1, -0.07, f'n={len(vals)}', ha='center', va='top',
            fontsize=8, color='#777', transform=ax.get_xaxis_transform())
    ax.text(1, -0.13, f'med={np.median(vals):.2f}', ha='center', va='top',
            fontsize=8, color='#333', fontweight='bold',
            transform=ax.get_xaxis_transform())

for i, (title, vals, color, gh_dots, vc) in enumerate(all_panels):
    draw_box(axes_flat[i], vals, color, title, gh_dots, violin_color=vc)

# Hide any unused axes
for j in range(len(all_panels), len(axes_flat)):
    axes_flat[j].set_visible(False)

# y-axis label only on leftmost of each row
for row in range(2):
    axes[row][0].set_ylabel('Relative citation score', fontsize=11)

fig.suptitle('Relative Citation Score: G+H together, Gummow, Hayne vs. each other judge',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/citation_all_judges_violinplot.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [22]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from matplotlib.patches import Patch

data = backup_cases_all

rows = []
for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    for p in case['paragraphs']:
        rows.append({'citation': citation, 'year': year,
                     'judge': p['judge'], 'citations': p.get('citations', 0)})
rows.sort(key=lambda x: -x['citations'])
top1000 = rows[:1000]

ALL_JUDGES = ['GUMMOW', 'HAYNE', 'GLEESON', 'CRENNAN', 'HEYDON',
              'FRENCH', 'KIEFEL', 'KIRBY', 'McHUGH', 'BELL',
              'CALLINAN', 'GAUDRON', 'BRENNAN']

YEARS = list(range(1998, 2013))
n_cols = 5
n_rows = 3   # 15 years = 3 rows x 5 cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, 13),
                         sharey=False)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

for idx, year in enumerate(YEARS):
    ax = axes_flat[idx]
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Paragraphs from this year only
    year_paras = [p for p in top1000 if p['year'] == year]

    # Count per judge and G+H co-authored paragraphs per judge
    judge_counts    = defaultdict(int)
    judge_gh_counts = defaultdict(int)
    gh_together     = 0
    for p in year_paras:
        names  = [n.strip() for n in p['judge'].split(',')]
        has_gh = 'GUMMOW' in names and 'HAYNE' in names
        for name in names:
            judge_counts[name] += 1
            if has_gh:
                judge_gh_counts[name] += 1
        if has_gh:
            gh_together += 1

    # Build bars: individual judges sorted by count, then G+H together
    judges  = sorted([j for j in ALL_JUDGES if judge_counts[j] > 0],
                     key=lambda j: -judge_counts[j])
    labels  = [j.capitalize() for j in judges] + ['G+H\ntogether']
    values  = [judge_counts[j] for j in judges] + [gh_together]
    colors  = ['#2980b9' if j in ('GUMMOW', 'HAYNE') else '#333333'
               for j in judges] + ['#1a5276']

    x = np.arange(len(labels))
    bars = ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5, zorder=3)

    # Red overlay: G+H co-authored portion for each individual judge bar
    gh_overlay = [judge_gh_counts.get(j, 0) for j in judges] + [0]
    ax.bar(x, gh_overlay, color='#cc3333', alpha=0.6,
           edgecolor='white', linewidth=0.5, zorder=4)

    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(values) * 0.02,
                    str(val), ha='center', va='bottom', fontsize=6.5, color='#222')

    # Divider before G+H bar
    if len(judges) > 0:
        ax.axvline(x=len(judges) - 0.5, color='#bbb', linewidth=0.8,
                   linestyle='--', zorder=2)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=6, rotation=40, ha='right')
    ax.set_title(f'{year}  (n={len(year_paras)})', fontsize=10,
                 fontweight='bold', pad=6)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', labelsize=7)

# y-axis label on leftmost of each row
for row in range(n_rows):
    axes[row][0].set_ylabel('Paragraphs in top 1000', fontsize=9)

legend_elements = [
    Patch(facecolor='#333333', label='Individual judge'),
    Patch(facecolor='#2980b9', label='Gummow / Hayne (individual)'),
    Patch(facecolor='#1a5276', label='Gummow & Hayne together'),
    Patch(facecolor='#cc3333', alpha=0.6, label='Portion written with G+H'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa',
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Top-1000 paragraphs per judge by year (1998–2012)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=2.5, w_pad=1.5)
plt.savefig('images/para_per_year_ghbluered.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [23]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from matplotlib.patches import Patch

data = backup_cases_all

rows = []
for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    for p in case['paragraphs']:
        rows.append({'citation': citation, 'year': year,
                     'judge': p['judge'], 'citations': p.get('citations', 0)})
rows.sort(key=lambda x: -x['citations'])
top1000 = rows[:1000]

ALL_JUDGES = ['GUMMOW', 'HAYNE', 'GLEESON', 'CRENNAN', 'HEYDON',
              'FRENCH', 'KIEFEL', 'KIRBY', 'McHUGH', 'BELL',
              'CALLINAN', 'GAUDRON', 'BRENNAN']

YEARS = list(range(1998, 2013))
n_cols = 5
n_rows = 3   # 15 years = 3 rows x 5 cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, 13),
                         sharey=False)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

for idx, year in enumerate(YEARS):
    ax = axes_flat[idx]
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Paragraphs from this year only
    year_paras = [p for p in top1000 if p['year'] == year]

    # Count per judge
    judge_counts = defaultdict(int)
    gh_together  = 0
    for p in year_paras:
        names = [n.strip() for n in p['judge'].split(',')]
        for name in names:
            judge_counts[name] += 1
        if 'GUMMOW' in names and 'HAYNE' in names:
            gh_together += 1

    # Build bars: individual judges sorted by count, then G+H together
    judges  = sorted([j for j in ALL_JUDGES if judge_counts[j] > 0],
                     key=lambda j: -judge_counts[j])
    labels  = [j.capitalize() for j in judges] + ['G+H\ntogether']
    values  = [judge_counts[j] for j in judges] + [gh_together]
    colors  = ['#2980b9' if j in ('GUMMOW', 'HAYNE') else '#333333'
               for j in judges] + ['#1a5276']

    x = np.arange(len(labels))
    bars = ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5, zorder=3)

    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(values) * 0.02,
                    str(val), ha='center', va='bottom', fontsize=6.5, color='#222')

    # Divider before G+H bar
    if len(judges) > 0:
        ax.axvline(x=len(judges) - 0.5, color='#bbb', linewidth=0.8,
                   linestyle='--', zorder=2)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=6, rotation=40, ha='right')
    ax.set_title(f'{year}  (n={len(year_paras)})', fontsize=10,
                 fontweight='bold', pad=6)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', labelsize=7)

# y-axis label on leftmost of each row
for row in range(n_rows):
    axes[row][0].set_ylabel('Paragraphs in top 1000', fontsize=9)

legend_elements = [
    Patch(facecolor='#333333', label='Individual judge'),
    Patch(facecolor='#2980b9', label='Gummow / Hayne (individual)'),
    Patch(facecolor='#1a5276', label='Gummow & Hayne together'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa',
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Top-1000 paragraphs per judge by year (1998–2012)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=2.5, w_pad=1.5)
plt.savefig('images/para_per_year_ghblue.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [24]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from matplotlib.patches import Patch

data = backup_cases_all

TOP99 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

# Build paragraph list from top 99 cases only
rows = []
for citation in TOP99:
    if citation not in data:
        continue
    year = int(citation.split(']')[0].replace('[', '').strip())
    for p in data[citation]['paragraphs']:
        rows.append({'citation': citation, 'year': year,
                     'judge': p['judge'], 'citations': p.get('citations', 0)})
rows.sort(key=lambda x: -x['citations'])

ALL_JUDGES = ['GUMMOW', 'HAYNE', 'GLEESON', 'CRENNAN', 'HEYDON',
              'FRENCH', 'KIEFEL', 'KIRBY', 'McHUGH', 'BELL',
              'CALLINAN', 'GAUDRON', 'BRENNAN']

YEARS  = list(range(1998, 2013))
n_cols = 5
n_rows = 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, 13), sharey=False)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

for idx, year in enumerate(YEARS):
    ax = axes_flat[idx]
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    year_paras = [p for p in rows if p['year'] == year]

    judge_counts = defaultdict(int)
    gh_together  = 0
    for p in year_paras:
        names = [n.strip() for n in p['judge'].split(',')]
        for name in names:
            judge_counts[name] += 1
        if 'GUMMOW' in names and 'HAYNE' in names:
            gh_together += 1

    judges  = sorted([j for j in ALL_JUDGES if judge_counts[j] > 0],
                     key=lambda j: -judge_counts[j])
    labels  = [j.capitalize() for j in judges] + ['G+H\ntogether']
    values  = [judge_counts[j] for j in judges] + [gh_together]
    colors  = ['#2980b9' if j in ('GUMMOW', 'HAYNE') else '#333333'
               for j in judges] + ['#1a5276']

    x    = np.arange(len(labels))
    bars = ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5, zorder=3)

    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(values) * 0.02,
                    str(val), ha='center', va='bottom', fontsize=6.5, color='#222')

    if len(judges) > 0:
        ax.axvline(x=len(judges) - 0.5, color='#bbb', linewidth=0.8,
                   linestyle='--', zorder=2)

    # n = cases from that year in top 99
    n_cases = len({p['citation'] for p in year_paras})
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=6, rotation=40, ha='right')
    ax.set_title(f'{year}  (n={n_cases} cases)', fontsize=10, fontweight='bold', pad=6)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', labelsize=7)

for row in range(n_rows):
    axes[row][0].set_ylabel('Paragraphs in top 99 cases', fontsize=9)

legend_elements = [
    Patch(facecolor='#333333', label='Individual judge'),
    Patch(facecolor='#2980b9', label='Gummow / Hayne (individual)'),
    Patch(facecolor='#1a5276', label='Gummow & Hayne together'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa',
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Paragraphs per judge by year — top 99 most cited cases (1998–2012)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=2.5, w_pad=1.5)
plt.savefig('images/para_per_year_top99.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [29]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from matplotlib.patches import Patch

data = backup_cases_all

TOP99 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

ALL_JUDGES = ['GUMMOW', 'HAYNE', 'GLEESON', 'CRENNAN', 'HEYDON',
              'FRENCH', 'KIEFEL', 'KIRBY', 'McHUGH', 'BELL',
              'CALLINAN', 'GAUDRON', 'BRENNAN']

# For each case, find the leading judgment and credit each author
yearly_judge = defaultdict(lambda: defaultdict(int))
yearly_gh    = defaultdict(int)
yearly_cases = defaultdict(int)

for citation in TOP99:
    if citation not in data:
        continue
    year    = int(citation.split(']')[0].replace('[', '').strip())
    case    = data[citation]
    leading = sorted(case['judgmentAuthors'],
                     key=lambda j: j.get('totalCitations', 0), reverse=True)[0]
    names   = [n.strip() for n in leading['name'].split(',')]

    yearly_cases[year] += 1
    for name in names:
        yearly_judge[year][name] += 1
    if 'GUMMOW' in names and 'HAYNE' in names:
        yearly_gh[year] += 1

YEARS  = list(range(1998, 2013))
n_cols = 5
n_rows = 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, 13), sharey=False)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

for idx, year in enumerate(YEARS):
    ax = axes_flat[idx]
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    judge_counts = yearly_judge[year]
    gh_together  = yearly_gh[year]

    judges  = sorted([j for j in ALL_JUDGES if judge_counts[j] > 0],
                     key=lambda j: -judge_counts[j])
    labels  = [j.capitalize() for j in judges] + ['G+H\ntogether']
    values  = [judge_counts[j] for j in judges] + [gh_together]
    colors  = ['#2980b9' if j in ('GUMMOW', 'HAYNE') else '#333333'
               for j in judges] + ['#1a5276']

    if not values or max(values) == 0:
        ax.set_title(f'{year}  (n=0)', fontsize=10, fontweight='bold', pad=6)
        continue

    x    = np.arange(len(labels))
    bars = ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5, zorder=3)

    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(values) * 0.02,
                    str(val), ha='center', va='bottom', fontsize=6.5, color='#222')

    if len(judges) > 0:
        ax.axvline(x=len(judges) - 0.5, color='#bbb', linewidth=0.8,
                   linestyle='--', zorder=2)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=6, rotation=40, ha='right')
    ax.set_title(f'{year}  (n={yearly_cases[year]} cases)', fontsize=10,
                 fontweight='bold', pad=6)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', labelsize=7)

for row in range(n_rows):
    axes[row][0].set_ylabel('Leading judgment appearances', fontsize=9)

legend_elements = [
    Patch(facecolor='#333333', label='Individual judge'),
    Patch(facecolor='#2980b9', label='Gummow / Hayne (individual)'),
    Patch(facecolor='#1a5276', label='Gummow & Hayne together'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa',
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Leading judgment authors by year — top 99 most cited cases (1998–2012)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=2.5, w_pad=1.5)
plt.savefig('images/leading_judgment_top99.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [31]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from matplotlib.patches import Patch

data = backup_cases_all

TOP99 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

ALL_JUDGES = ['GUMMOW', 'HAYNE', 'GLEESON', 'CRENNAN', 'HEYDON',
              'FRENCH', 'KIEFEL', 'KIRBY', 'McHUGH', 'BELL',
              'CALLINAN', 'GAUDRON', 'BRENNAN']

judge_counts = defaultdict(int)
gh_together  = 0

for citation in TOP99:
    if citation not in data:
        continue
    case    = data[citation]
    leading = sorted(case['judgmentAuthors'],
                     key=lambda j: j.get('totalCitations', 0), reverse=True)[0]
    names   = [n.strip() for n in leading['name'].split(',')]
    for name in names:
        judge_counts[name] += 1
    if 'GUMMOW' in names and 'HAYNE' in names:
        gh_together += 1

judges  = sorted([j for j in ALL_JUDGES if judge_counts[j] > 0],
                 key=lambda j: -judge_counts[j])
labels  = [j.capitalize() for j in judges] + ['G+H\ntogether']
values  = [judge_counts[j] for j in judges] + [gh_together]
colors  = ['#2980b9' if j in ('GUMMOW', 'HAYNE') else '#333333'
           for j in judges] + ['#1a5276']

x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

bars = ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.6, zorder=3)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.01,
            str(val), ha='center', va='bottom', fontsize=9, color='#222')

ax.axvline(x=len(judges) - 0.5, color='#bbb', linewidth=1,
           linestyle='--', zorder=2)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Leading judgment appearances', fontsize=11)
ax.set_title('Leading judgment authors — top 99 most cited cases)',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_elements = [
    Patch(facecolor='#333333', label='Individual judge'),
    Patch(facecolor='#2980b9', label='Gummow / Hayne (individual)'),
    Patch(facecolor='#1a5276', label='Gummow & Hayne together'),
]
ax.legend(handles=legend_elements, fontsize=10, frameon=True,
          facecolor='white', edgecolor='#aaa')

plt.tight_layout()
plt.savefig('images/leading_judgment_combined.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [28]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from matplotlib.patches import Patch

data = backup_cases_all

rows = []
for citation, case in data.items():
    for p in case['paragraphs']:
        rows.append({'judge': p['judge'], 'citations': p.get('citations', 0)})
rows.sort(key=lambda x: -x['citations'])
top1000 = rows[:1000]

ALL_JUDGES = ['GUMMOW', 'HAYNE', 'GLEESON', 'CRENNAN', 'HEYDON',
              'FRENCH', 'KIEFEL', 'KIRBY', 'McHUGH', 'BELL',
              'CALLINAN', 'GAUDRON', 'BRENNAN']

judge_counts = defaultdict(int)
gh_together  = 0

for p in top1000:
    names = [n.strip() for n in p['judge'].split(',')]
    for name in names:
        judge_counts[name] += 1
    if 'GUMMOW' in names and 'HAYNE' in names:
        gh_together += 1

judges  = sorted([j for j in ALL_JUDGES if judge_counts[j] > 0],
                 key=lambda j: -judge_counts[j])
labels  = [j.capitalize() for j in judges] + ['G+H\ntogether']
values  = [judge_counts[j] for j in judges] + [gh_together]
colors  = ['#2980b9' if j in ('GUMMOW', 'HAYNE') else '#333333'
           for j in judges] + ['#1a5276']

x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f8f8')

bars = ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.6, zorder=3)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.01,
            str(val), ha='center', va='bottom', fontsize=9, color='#222')

ax.axvline(x=len(judges) - 0.5, color='#bbb', linewidth=1, linestyle='--', zorder=2)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Paragraphs in top 1000', fontsize=11)
ax.set_title('Who wrote the top 1000 most cited paragraphs?',
             fontsize=13, fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_elements = [
    Patch(facecolor='#333333', label='Individual judge'),
    Patch(facecolor='#2980b9', label='Gummow / Hayne (individual)'),
    Patch(facecolor='#1a5276', label='Gummow & Hayne together'),
]
ax.legend(handles=legend_elements, fontsize=10, frameon=True,
          facecolor='white', edgecolor='#aaa')

plt.tight_layout()
plt.savefig('images/top1000_para_authors.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [32]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from matplotlib.patches import Patch

data = backup_cases_all

ALL_JUDGES = ['GUMMOW', 'HAYNE', 'GLEESON', 'CRENNAN', 'HEYDON',
              'FRENCH', 'KIEFEL', 'KIRBY', 'McHUGH', 'BELL',
              'CALLINAN', 'GAUDRON', 'BRENNAN', 'TOOHEY']

YEARS = list(range(1998, 2013))

yearly_judge  = defaultdict(lambda: defaultdict(int))
yearly_gh     = defaultdict(int)
yearly_cases  = defaultdict(int)

for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    if year not in YEARS:
        continue
    leading = sorted(case['judgmentAuthors'],
                     key=lambda j: j.get('totalCitations', 0), reverse=True)[0]
    names = [n.strip() for n in leading['name'].split(',')]
    yearly_cases[year] += 1
    for name in names:
        yearly_judge[year][name] += 1
    if 'GUMMOW' in names and 'HAYNE' in names:
        yearly_gh[year] += 1

n_cols = 5
n_rows = 3

fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, 13), sharey=False)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

for idx, year in enumerate(YEARS):
    ax = axes_flat[idx]
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    judge_counts = yearly_judge[year]
    gh_together  = yearly_gh[year]

    judges  = sorted([j for j in ALL_JUDGES if judge_counts[j] > 0],
                     key=lambda j: -judge_counts[j])
    labels  = [j.capitalize() for j in judges] + ['G+H\ntogether']
    values  = [judge_counts[j] for j in judges] + [gh_together]
    colors  = ['#2980b9' if j in ('GUMMOW', 'HAYNE') else '#333333'
               for j in judges] + ['#1a5276']

    if not values or max(values) == 0:
        ax.set_title(f'{year}  (n=0)', fontsize=10, fontweight='bold', pad=6)
        continue

    x    = np.arange(len(labels))
    bars = ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5, zorder=3)

    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(values) * 0.02,
                    str(val), ha='center', va='bottom', fontsize=6.5, color='#222')

    if len(judges) > 0:
        ax.axvline(x=len(judges) - 0.5, color='#bbb', linewidth=0.8,
                   linestyle='--', zorder=2)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=6, rotation=40, ha='right')
    ax.set_title(f'{year}  (n={yearly_cases[year]} cases)', fontsize=10,
                 fontweight='bold', pad=6)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', labelsize=7)

for row in range(n_rows):
    axes[row][0].set_ylabel('Leading judgment appearances', fontsize=9)

legend_elements = [
    Patch(facecolor='#333333', label='Individual judge'),
    Patch(facecolor='#2980b9', label='Gummow / Hayne (individual)'),
    Patch(facecolor='#1a5276', label='Gummow & Hayne together'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa',
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Leading judgment authors by year — all cases (1998–2012)\n'
             '(each judge credited for every leading judgment they co-authored)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(h_pad=2.5, w_pad=1.5)
plt.savefig('images/leading_judgment_all_years.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [ ]:
import json
import csv

data = backup_cases_all

majority_judgments = []

for citation, case in data.items():
    bench_size = len(case.get('bench', []))
    bench      = ', '.join(case.get('bench', []))
    tags       = ', '.join(case.get('law_areas', []))
    year       = int(citation.split(']')[0].replace('[', '').strip())

    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        if bench_size > 0 and len(names) > bench_size / 2:
            majority_judgments.append({
                'citation':         citation,
                'year':             year,
                'tags':             tags,
                'judgment_authors': j['name'],
                'n_authors':        len(names),
                'bench_size':       bench_size,
                'bench':            bench,
                'total_citations':  j.get('totalCitations', 0),
                'relative_citation': j.get('relativeCitation', ''),
            })

# Sort by citations descending
majority_judgments.sort(key=lambda x: -x['total_citations'])

"""print(f"Total majority judgments: {len(majority_judgments)}")
print()
print(f"{'Rank':<5} {'Citation':<20} {'Year':<6} {'Authors':<50} {'n/bench':<10} {'Citations':>10}")
print("-" * 110)
for i, m in enumerate(majority_judgments, 1):
    print(f"{i:<5} {m['citation']:<20} {m['year']:<6} {m['judgment_authors']:<50} "
          f"{m['n_authors']}/{m['bench_size']:<8} {m['total_citations']:>10,}")

# Save CSV
with open('majority_judgments.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(majority_judgments[0].keys()))
    writer.writeheader()
    writer.writerows(majority_judgments)

print(f"\nSaved: majority_judgments.csv ({len(majority_judgments)} rows)")"""

In [33]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from matplotlib.patches import Patch

data = backup_cases_all

total_cases = len(data)
cases_with_majority = 0
gh_majority_cases   = 0
tag_total        = defaultdict(int)
tag_majority     = defaultdict(int)
tag_gh_majority  = defaultdict(int)
judge_majority   = defaultdict(int)
judge_total      = defaultdict(int)

for citation, case in data.items():
    bench_size = len(case.get('bench', []))
    tags = case.get('law_areas', [])
    for tag in tags:
        tag_total[tag] += 1
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            judge_total[name] += 1
    has_majority = False
    has_gh_majority = False
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        if bench_size > 0 and len(names) > bench_size / 2:
            has_majority = True
            for name in names:
                judge_majority[name] += 1
            if 'GUMMOW' in names and 'HAYNE' in names:
                has_gh_majority = True
    if has_majority:
        cases_with_majority += 1
        for tag in tags:
            tag_majority[tag] += 1
    if has_gh_majority:
        gh_majority_cases += 1
        for tag in tags:
            tag_gh_majority[tag] += 1

GH_BLUE   = '#1a5276'
MID_BLUE  = '#2980b9'
DARK      = '#333333'
LIGHT     = '#aaaaaa'
RED       = '#cc3333'

fig = plt.figure(figsize=(16, 14))
fig.patch.set_facecolor('white')

# ── 1. Pie ─────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 2, 1)
ax1.set_facecolor('white')
no_majority = total_cases - cases_with_majority
gh_only     = gh_majority_cases
maj_no_gh   = cases_with_majority - gh_majority_cases
sizes  = [gh_only, maj_no_gh, no_majority]
colors = [GH_BLUE, DARK, LIGHT]
labels = [
    f'Majority incl. G+H\n({gh_only}, {gh_only/total_cases:.1%})',
    f'Majority excl. G+H\n({maj_no_gh}, {maj_no_gh/total_cases:.1%})',
    f'No majority\n({no_majority}, {no_majority/total_cases:.1%})',
]
ax1.pie(sizes, labels=labels, colors=colors, startangle=90,
        textprops={'fontsize': 9})
ax1.set_title('Proportion of cases with a majority judgment', fontsize=11, fontweight='bold', pad=12)

# ── 2. Horizontal bar: majority rate by tag ────────────────────────────
ax2 = fig.add_subplot(2, 2, 2)
ax2.set_facecolor('#f8f8f8')
tags_sorted = sorted(tag_total, key=lambda t: tag_majority.get(t, 0) / tag_total[t])
rates    = [tag_majority.get(t, 0) / tag_total[t] for t in tags_sorted]
gh_rates = [tag_gh_majority.get(t, 0) / tag_total[t] for t in tags_sorted]
totals   = [tag_total[t] for t in tags_sorted]
y = np.arange(len(tags_sorted))

ax2.barh(y, rates,    color=DARK,    edgecolor='white', linewidth=0.5, zorder=3, label='Majority (all)')
ax2.barh(y, gh_rates, color=GH_BLUE, edgecolor='white', linewidth=0.5, zorder=4, alpha=0.85,
         label='Majority incl. G+H')
for i, (r, n) in enumerate(zip(rates, totals)):
    ax2.text(r + 0.005, i, f'{r:.0%}  (n={n})', va='center', fontsize=7, color='#333')
ax2.set_yticks(y)
ax2.set_yticklabels([t.replace(' & ', '\n& ') for t in tags_sorted], fontsize=7.5)
ax2.set_xlabel('Proportion of cases', fontsize=10)
ax2.set_title('Majority judgment rate by tag', fontsize=11, fontweight='bold', pad=12)
ax2.set_xlim(0, 1.3)
ax2.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
ax2.xaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax2.set_axisbelow(True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.legend(fontsize=8.5, frameon=True, facecolor='white', edgecolor='#aaa', loc='lower right')

# ── 3. Bar: judge majority appearances (raw) ───────────────────────────
ax3 = fig.add_subplot(2, 2, 3)
ax3.set_facecolor('#f8f8f8')
judges_sorted = sorted(judge_majority, key=lambda j: -judge_majority[j])
jvals   = [judge_majority[j] for j in judges_sorted]
jlabels = [j.capitalize() for j in judges_sorted]
jcolors = [MID_BLUE if j in ('GUMMOW', 'HAYNE') else DARK for j in judges_sorted]
x = np.arange(len(jlabels))
ax3.bar(x, jvals, color=jcolors, edgecolor='white', linewidth=0.5, zorder=3)
for i, v in enumerate(jvals):
    ax3.text(i, v + 2, str(v), ha='center', va='bottom', fontsize=8, color='#222')
ax3.set_xticks(x)
ax3.set_xticklabels(jlabels, fontsize=8, rotation=40, ha='right')
ax3.set_ylabel('Majority judgment appearances', fontsize=10)
ax3.set_title('Judge majority appearances (raw)', fontsize=11, fontweight='bold', pad=12)
ax3.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax3.set_axisbelow(True)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
legend_elements = [Patch(facecolor=MID_BLUE, label='Gummow / Hayne'),
                   Patch(facecolor=DARK,     label='Other judges')]
ax3.legend(handles=legend_elements, fontsize=8.5, frameon=True, facecolor='white', edgecolor='#aaa')

# ── 4. Bar: judge majority rate (normalised) ───────────────────────────
ax4 = fig.add_subplot(2, 2, 4)
ax4.set_facecolor('#f8f8f8')
rates_j = [judge_majority[j] / judge_total[j] for j in judges_sorted]
ax4.bar(x, rates_j, color=jcolors, edgecolor='white', linewidth=0.5, zorder=3)
for i, v in enumerate(rates_j):
    ax4.text(i, v + 0.005, f'{v:.0%}', ha='center', va='bottom', fontsize=8, color='#222')
ax4.set_xticks(x)
ax4.set_xticklabels(jlabels, fontsize=8, rotation=40, ha='right')
ax4.set_ylabel('% of judgments in majority', fontsize=10)
ax4.set_title('Judge majority rate (% of their judgments)', fontsize=11, fontweight='bold', pad=12)
ax4.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
ax4.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax4.set_axisbelow(True)
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)
ax4.legend(handles=legend_elements, fontsize=8.5, frameon=True, facecolor='white', edgecolor='#aaa')

fig.suptitle('Majority Judgment Statistics', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(pad=2.5)
plt.savefig('images/majority_stats.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [34]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from collections import defaultdict
from itertools import combinations

data = backup_cases_all

comat        = defaultdict(int)
judge_totals = defaultdict(int)

for case in data.values():
    bench_size = len(case.get('bench', []))
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        if bench_size > 0 and len(names) > bench_size / 2:
            for name in names:
                judge_totals[name] += 1
            for a, b in combinations(sorted(names), 2):
                comat[(a, b)] += 1

JUDGES = list(judge_totals.keys())

# ── Graph 1: raw ─────────────────────────────────────────────────────────
def build_and_draw(weight_fn, title, filename, edge_scale=5.5):
    G = nx.Graph()
    G.add_nodes_from(JUDGES)
    for (a, b), count in comat.items():
        w = weight_fn(a, b, count)
        if w > 0:
            G.add_edge(a, b, weight=w, raw=count)

    weights = [G[u][v]['weight'] for u, v in G.edges()]
    w_min, w_max = min(weights), max(weights)
    pos = nx.spring_layout(G, weight='weight', seed=42, k=3.2)

    fig, ax = plt.subplots(figsize=(12, 9))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.axis('off')

    for u, v in G.edges():
        w = G[u][v]['weight']
        t = (w - w_min) / (w_max - w_min) if w_max > w_min else 1.0
        alpha = 0.15 + t * 0.75
        lw    = 0.4 + t * edge_scale
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        ax.plot([x0, x1], [y0, y1], color='#666666', alpha=alpha,
                linewidth=lw, zorder=1, solid_capstyle='round')

    max_total = max(judge_totals[j] for j in JUDGES)
    for judge in JUDGES:
        x, y   = pos[judge]
        size   = 300 + 1400 * (judge_totals[judge] / max_total)
        color  = '#2980b9' if judge in ('GUMMOW', 'HAYNE') else '#333333'
        offset = 0.055 + 0.04 * (judge_totals[judge] / max_total)
        ax.scatter(x, y, s=size, color=color, zorder=3,
                   edgecolors='#888888', linewidths=1.5)
        ax.text(x, y + offset, f"{judge.capitalize()} ({judge_totals[judge]})",
                ha='center', va='bottom', fontsize=9,
                color='#111111', fontweight='bold', zorder=5)

    for label, t in [('Weak', 0.05), ('Medium', 0.5), ('Strong', 1.0)]:
        lw    = 0.4 + t * edge_scale
        alpha = 0.15 + t * 0.75
        ax.plot([], [], color='#666666', alpha=alpha, linewidth=lw, label=label)
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2980b9', label='Gummow / Hayne'),
        Patch(facecolor='#333333', label='Other judges'),
    ]
    ax.legend(handles=legend_elements, loc='lower left', fontsize=9,
              frameon=True, facecolor='white', edgecolor='#aaa')

    ax.set_title(title, fontsize=13, fontweight='bold', color='#111111', pad=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

# Raw
build_and_draw(
    lambda a, b, c: c,
    title='Majority Judgment Co-authorship Network (raw count)',
    filename='images/majority_network_raw.png',
    edge_scale=5.5
)

# Normalised: 2C / (A+B)
build_and_draw(
    lambda a, b, c: (2 * c) / (judge_totals[a] + judge_totals[b]),
    title='Majority Judgment Co-authorship Network (normalised by output)',
    filename='images/majority_network_normalised.png',
    edge_scale=6.0
)

Saved: images/majority_network_raw.png
Saved: images/majority_network_normalised.png


In [42]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import networkx as nx
import numpy as np
from collections import defaultdict
from itertools import combinations

data = backup_cases_all

YEARS  = list(range(1998, 2013))
n_cols = 3
n_rows = 5

yearly_comat  = defaultdict(lambda: defaultdict(int))
yearly_totals = defaultdict(lambda: defaultdict(int))

for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    if year not in YEARS:
        continue
    bench_size = len(case.get('bench', []))
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        if bench_size > 0 and len(names) > bench_size / 2:
            for name in names:
                yearly_totals[year][name] += 1
            for a, b in combinations(sorted(names), 2):
                yearly_comat[year][(a, b)] += 1

# All judges ever active — fixed positions
ALL_JUDGES = sorted({n for y in yearly_totals.values() for n in y})

# Fixed positions: use circular layout so every judge always in same spot
pos = nx.circular_layout(nx.path_graph(len(ALL_JUDGES)))
pos = {judge: pos[i] for i, judge in enumerate(ALL_JUDGES)}

# Global scales
global_max      = max(c for y in YEARS for c in yearly_comat[y].values()) if any(yearly_comat[y] for y in YEARS) else 1
global_max_node = max(c for y in YEARS for c in yearly_totals[y].values()) if any(yearly_totals[y] for y in YEARS) else 1

# Colourmap: yellow → blue
cmap_edge = plt.cm.YlGnBu

def edge_color(t):
    return cmap_edge(t)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(30, 50))
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

x_all = [p[0] for p in pos.values()]
y_all = [p[1] for p in pos.values()]
x_pad, y_pad = 0.25, 0.28

for idx, year in enumerate(YEARS):
    ax = axes_flat[idx]
    ax.set_facecolor('#f5f5f5')
    ax.axis('off')
    ax.set_xlim(min(x_all) - x_pad, max(x_all) + x_pad)
    ax.set_ylim(min(y_all) - y_pad, max(y_all) + y_pad)

    comat  = yearly_comat[year]
    totals = yearly_totals[year]

    # Edges
    for (a, b), c in comat.items():
        t     = c / global_max
        alpha = 0.12 + t * 0.85
        lw    = 0.5 + t * 10.0
        x0, y0 = pos[a]
        x1, y1 = pos[b]
        ax.plot([x0, x1], [y0, y1], color=edge_color(t), alpha=alpha,
                linewidth=lw, zorder=1, solid_capstyle='round')

    # Nodes — all judges in fixed positions; grey out inactive ones
    for judge in ALL_JUDGES:
        x, y   = pos[judge]
        active = judge in totals
        size   = (200 + 1200 * (totals[judge] / global_max_node)) if active else 120
        color  = '#333333' if active else '#cccccc'
        offset = (0.08 + 0.05 * (totals[judge] / global_max_node)) if active else 0.08
        ec     = '#888' if active else '#ccc'
        ax.scatter(x, y, s=size, color=color, zorder=3, edgecolors=ec, linewidths=1.2)
        ax.text(x, y + offset, judge.capitalize(), ha='center', va='bottom',
                fontsize=11, color='#111' if active else '#bbb',
                fontweight='bold' if active else 'normal', zorder=4)

    ax.set_title(f'{year}', fontsize=30, fontweight='bold', pad=12, color='#111')

# Colour scale bar at bottom
fig.subplots_adjust(bottom=0.04)
cbar_ax = fig.add_axes([0.25, 0.01, 0.5, 0.008])
sm = plt.cm.ScalarMappable(
    cmap=plt.cm.YlGnBu,
    norm=mcolors.Normalize(vmin=0, vmax=global_max)
)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cbar.set_label(f'Co-authored majority judgments  (low = 0, high = {global_max})', fontsize=30)
cbar.set_ticks([0, global_max // 2, global_max])
cbar.set_ticklabels([f'0', f'{global_max//2}', f'{global_max}'], fontsize=30)

fig.suptitle('Majority Judgment Co-authorship Network by Year (1998–2012)\n'
             '(inactive judges shown in grey; edge colour = co-authorship count)',
             fontsize=35, fontweight='bold', y=1.005)
plt.tight_layout(h_pad=1.5, w_pad=1.0, rect=[0, 0.03, 1, 1])
plt.savefig('images/majority_network_years.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

/var/folders/4k/yfp_3j857vq5fj717924mjfm0000gn/T/ipykernel_30359/1073663308.py:110: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(h_pad=1.5, w_pad=1.0, rect=[0, 0.03, 1, 1])


Saved.


In [43]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict, Counter

data = backup_cases_all 

third_judges    = Counter()
judge_maj_total = defaultdict(int)

for case in data.values():
    bench_size = len(case.get('bench', []))
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        if bench_size > 0 and len(names) > bench_size / 2:
            has_g = 'GUMMOW' in names
            has_h = 'HAYNE'  in names
            for name in names:
                judge_maj_total[name] += 1
                if has_g and has_h and name not in ('GUMMOW', 'HAYNE'):
                    third_judges[name] += 1

# Sort by raw count
judges  = [j for j, _ in third_judges.most_common()]
raw     = [third_judges[j] for j in judges]
normed  = [third_judges[j] / judge_maj_total[j] for j in judges]
labels  = [j.capitalize() for j in judges]
x       = np.arange(len(judges))

DARK    = '#333333'
GH_BLUE = '#1a5276'
RED     = '#cc3333'

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# --- Raw ---
ax1 = axes[0]
ax1.set_facecolor('#f8f8f8')
bars = ax1.bar(x, raw, color=DARK, edgecolor='white', linewidth=0.6, zorder=3)
for bar, val in zip(bars, raw):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             str(val), ha='center', va='bottom', fontsize=9, color='#222')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=40, ha='right', fontsize=9.5)
ax1.set_ylabel('Times in majority judgment with G+H', fontsize=11)
ax1.set_title('Who writes majority judgments with\nGummow & Hayne? (raw)', fontsize=12, fontweight='bold', pad=10)
ax1.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax1.set_axisbelow(True)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Normalised ---
ax2 = axes[1]
ax2.set_facecolor('#f8f8f8')
# Sort by normalised rate for this panel
sort_idx = sorted(range(len(judges)), key=lambda i: -normed[i])
judges_n = [judges[i] for i in sort_idx]
normed_n = [normed[i] for i in sort_idx]
raw_n    = [raw[i]    for i in sort_idx]
labels_n = [j.capitalize() for j in judges_n]
x2 = np.arange(len(judges_n))

bars2 = ax2.bar(x2, normed_n, color=GH_BLUE, edgecolor='white', linewidth=0.6, zorder=3)
for bar, val, r in zip(bars2, normed_n, raw_n):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f'{val:.0%}', ha='center', va='bottom', fontsize=9, color='#222')
ax2.set_xticks(x2)
ax2.set_xticklabels(labels_n, rotation=40, ha='right', fontsize=9.5)
ax2.set_ylabel('% of judge\'s majority judgments that included G+H', fontsize=11)
ax2.set_title('Who writes majority judgments with\nGummow & Hayne? (normalised)', fontsize=12, fontweight='bold', pad=10)
ax2.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
ax2.set_ylim(0, 1.0)
ax2.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax2.set_axisbelow(True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle('Judges most likely to write majority judgments with Gummow & Hayne',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(pad=2.5)
plt.savefig('images/gh_majority_judges.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [46]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from itertools import combinations

data = backup_cases_all

YEARS = list(range(1998, 2013))

yearly_comat  = defaultdict(lambda: defaultdict(int))
yearly_totals = defaultdict(lambda: defaultdict(int))
yearly_active = defaultdict(set)

for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    if year not in YEARS:
        continue
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            yearly_totals[year][name] += 1
            yearly_active[year].add(name)
        for a, b in combinations(sorted(names), 2):
            yearly_comat[year][(a, b)] += 1

# Global vmax for consistent colorscale across both images
all_raw, all_norm = [], []
for year in YEARS:
    for (a, b), c in yearly_comat[year].items():
        all_raw.append(c)
        ta = yearly_totals[year].get(a, 0)
        tb = yearly_totals[year].get(b, 0)
        if ta + tb > 0:
            all_norm.append((2 * c) / (ta + tb))
raw_vmax  = max(all_raw)  if all_raw  else 1
norm_vmax = max(all_norm) if all_norm else 1

def draw_heatmap(ax, matrix, labels, vmin, vmax, cmap, fmt=None):
    n = len(labels)
    off_diag = matrix.copy()
    np.fill_diagonal(off_diag, 0)
    im = ax.imshow(off_diag, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    for i in range(n):
        for j in range(n):
            val = matrix[i, j]
            if i == j:
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                                            facecolor='#bbbbbb', zorder=1))
                if val > 0:
                    txt = str(int(val)) if fmt is None else f'{val:.2f}'
                    ax.text(j, i, txt, ha='center', va='center',
                            fontsize=7, color='#333', fontweight='bold', zorder=2)
            elif val > 0:
                txt = str(int(val)) if fmt is None else f'{val:.2f}'
                tc  = 'white' if off_diag[i, j] > vmax * 0.6 else '#333'
                ax.text(j, i, txt, ha='center', va='center',
                        fontsize=7, color=tc, zorder=2)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    return im

def build_year_data(year):
    active      = yearly_active[year]
    year_judges = sorted(active, key=lambda j: -yearly_totals[year].get(j, 0))
    n_y         = len(year_judges)
    idx_y       = {j: i for i, j in enumerate(year_judges)}
    labels_y    = [j.capitalize() for j in year_judges]

    raw_mat  = np.zeros((n_y, n_y))
    norm_mat = np.zeros((n_y, n_y))

    for i, ji in enumerate(year_judges):
        raw_mat[i, i]  = yearly_totals[year].get(ji, 0)
        norm_mat[i, i] = yearly_totals[year].get(ji, 0)

    for (a, b), c in yearly_comat[year].items():
        if a in idx_y and b in idx_y:
            ia, ib = idx_y[a], idx_y[b]
            raw_mat[ia, ib] = c
            raw_mat[ib, ia] = c
            ta  = yearly_totals[year].get(a, 0)
            tb  = yearly_totals[year].get(b, 0)
            val = (2 * c) / (ta + tb) if (ta + tb) > 0 else 0
            norm_mat[ia, ib] = val
            norm_mat[ib, ia] = val

    return labels_y, raw_mat, norm_mat, n_y

def make_combined_image(years, filename, title):
    # Pair up years into rows: [yr1, yr2], [yr3, yr4], ...
    rows = []
    for i in range(0, len(years), 2):
        rows.append(years[i:i+2])

    n_rows = len(rows)
    fig, axes = plt.subplots(n_rows, 4, figsize=(44, n_rows * 7))
    fig.patch.set_facecolor('white')
    fig.suptitle(title, fontsize=22, fontweight='bold', y=1.01)

    # Handle single-row case
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row_idx, year_pair in enumerate(rows):
        for col_pair, year in enumerate(year_pair):
            labels_y, raw_mat, norm_mat, n_y = build_year_data(year)
            col_raw  = col_pair * 2
            col_norm = col_pair * 2 + 1

            im1 = draw_heatmap(axes[row_idx][col_raw],  raw_mat,  labels_y, 0, raw_vmax,  'YlOrRd')
            axes[row_idx][col_raw].set_title(f'{year} — Raw',
                                fontsize=15, fontweight='bold', pad=8)
            plt.colorbar(im1, ax=axes[row_idx][col_raw], shrink=0.85, label='Co-authored judgments')

            im2 = draw_heatmap(axes[row_idx][col_norm], norm_mat, labels_y, 0, norm_vmax, 'YlOrRd', fmt='norm')
            axes[row_idx][col_norm].set_title(f'{year} — Normalised (2C / A+B)',
                                fontsize=15, fontweight='bold', pad=8)
            plt.colorbar(im2, ax=axes[row_idx][col_norm], shrink=0.85, label='Normalised score')

        # Hide unused axes if odd number of years
        if len(year_pair) == 1:
            axes[row_idx][2].set_visible(False)
            axes[row_idx][3].set_visible(False)

    plt.tight_layout()
    plt.savefig(filename, dpi=130, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

make_combined_image(
    years=list(range(1998, 2006)),
    filename='images/heatmaps_1998_2005.png',
    title='Judge Co-authorship Heatmaps 1998–2005'
)

make_combined_image(
    years=list(range(2006, 2013)),
    filename='images/heatmaps_2006_2012.png',
    title='Judge Co-authorship Heatmaps 2006–2012'
)

Saved: images/heatmaps_1998_2005.png
Saved: images/heatmaps_2006_2012.png


In [54]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from itertools import combinations

data = backup_cases_all

YEARS = list(range(1998, 2013))

yearly_comat  = defaultdict(lambda: defaultdict(int))
yearly_totals = defaultdict(lambda: defaultdict(int))
yearly_active = defaultdict(set)

for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    if year not in YEARS:
        continue
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            yearly_totals[year][name] += 1
            yearly_active[year].add(name)
        for a, b in combinations(sorted(names), 2):
            yearly_comat[year][(a, b)] += 1

all_raw, all_norm = [], []
for year in YEARS:
    for (a, b), c in yearly_comat[year].items():
        all_raw.append(c)
        ta = yearly_totals[year].get(a, 0)
        tb = yearly_totals[year].get(b, 0)
        if ta + tb > 0:
            all_norm.append((2 * c) / (ta + tb))
raw_vmax  = max(all_raw)  if all_raw  else 1
norm_vmax = max(all_norm) if all_norm else 1

def draw_heatmap(ax, matrix, labels, vmin, vmax, cmap, fmt=None):
    n = len(labels)
    off_diag = matrix.copy()
    np.fill_diagonal(off_diag, 0)
    im = ax.imshow(off_diag, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    for i in range(n):
        for j in range(n):
            val = matrix[i, j]
            if i == j:
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                                            facecolor='#bbbbbb', zorder=1))
                if val > 0:
                    txt = str(int(val)) if fmt is None else f'{val:.2f}'
                    ax.text(j, i, txt, ha='center', va='center',
                            fontsize=14, color='#333', fontweight='bold', zorder=2)
            elif val > 0:
                txt = str(int(val)) if fmt is None else f'{val:.2f}'
                tc  = 'white' if off_diag[i, j] > vmax * 0.6 else '#333'
                ax.text(j, i, txt, ha='center', va='center',
                        fontsize=14, color=tc, zorder=2)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=14)
    ax.set_yticklabels(labels, fontsize=14)
    return im

def build_year_data(year):
    active      = yearly_active[year]
    year_judges = sorted(active, key=lambda j: -yearly_totals[year].get(j, 0))
    n_y         = len(year_judges)
    idx_y       = {j: i for i, j in enumerate(year_judges)}
    labels_y    = [j.capitalize() for j in year_judges]
    raw_mat     = np.zeros((n_y, n_y))
    norm_mat    = np.zeros((n_y, n_y))
    for i, ji in enumerate(year_judges):
        raw_mat[i, i]  = yearly_totals[year].get(ji, 0)
        norm_mat[i, i] = yearly_totals[year].get(ji, 0)
    for (a, b), c in yearly_comat[year].items():
        if a in idx_y and b in idx_y:
            ia, ib = idx_y[a], idx_y[b]
            raw_mat[ia, ib] = c
            raw_mat[ib, ia] = c
            ta  = yearly_totals[year].get(a, 0)
            tb  = yearly_totals[year].get(b, 0)
            val = (2 * c) / (ta + tb) if (ta + tb) > 0 else 0
            norm_mat[ia, ib] = val
            norm_mat[ib, ia] = val
    return labels_y, raw_mat, norm_mat, n_y

def make_combined_image(years, filename, title):
    n_years = len(years)
    fig, axes = plt.subplots(n_years, 2, figsize=(26, n_years * 9))
    fig.patch.set_facecolor('white')
    fig.suptitle(title, fontsize=30, fontweight='bold', y=1.01)

    if n_years == 1:
        axes = axes[np.newaxis, :]

    for row, year in enumerate(years):
        labels_y, raw_mat, norm_mat, n_y = build_year_data(year)
        draw_heatmap(axes[row][0], raw_mat,  labels_y, 0, raw_vmax,  'YlOrRd')
        axes[row][0].set_title(f'{year} — Raw co-authored judgments',
                               fontsize=18, fontweight='bold', pad=12)
        draw_heatmap(axes[row][1], norm_mat, labels_y, 0, norm_vmax, 'YlOrRd', fmt='norm')
        axes[row][1].set_title(f'{year} — Normalised (2C / A+B)',
                               fontsize=18, fontweight='bold', pad=12)

    plt.tight_layout(rect=[0, 0.04, 1, 1])

    # One shared colourbar per type
    cbar_ax1 = fig.add_axes([0.08, 0.01, 0.38, 0.012])
    sm1 = plt.cm.ScalarMappable(cmap='YlOrRd', norm=plt.Normalize(vmin=0, vmax=raw_vmax))
    sm1.set_array([])
    cb1 = fig.colorbar(sm1, cax=cbar_ax1, orientation='horizontal')
    cb1.set_label('Co-authored judgments (raw)', fontsize=14)
    cb1.ax.tick_params(labelsize=14)

    cbar_ax2 = fig.add_axes([0.55, 0.01, 0.38, 0.012])
    sm2 = plt.cm.ScalarMappable(cmap='YlOrRd', norm=plt.Normalize(vmin=0, vmax=norm_vmax))
    sm2.set_array([])
    cb2 = fig.colorbar(sm2, cax=cbar_ax2, orientation='horizontal')
    cb2.set_label('Normalised score (2C / A+B)', fontsize=14)
    cb2.ax.tick_params(labelsize=14)

    plt.savefig(filename, dpi=130, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

make_combined_image(list(range(1998, 2003)), 'images/heatmaps_1998_2002.png', 'Judge Co-authorship Heatmaps 1998–2002')
make_combined_image(list(range(2003, 2008)), 'images/heatmaps_2003_2007.png', 'Judge Co-authorship Heatmaps 2003–2007')
make_combined_image(list(range(2008, 2013)), 'images/heatmaps_2008_2012.png', 'Judge Co-authorship Heatmaps 2008–2012')


Saved: images/heatmaps_1998_2002.png
Saved: images/heatmaps_2003_2007.png
Saved: images/heatmaps_2008_2012.png


In [ ]:

# ── Cleanup: remove 3 cases with known bench/judge issues ─────────────────────────
import json

with open('cases_all_updated.json') as f:
    data = json.load(f)

for citation in ['[2007] HCA 58', '[1997] HCA 50', '[1997] HCA 51']:
    data.pop(citation, None)

with open('cases_all_updated.json', 'w') as f:
    json.dump(data, f, indent=2)

print(f"Remaining cases: {len(data)}")

Remaining cases: 778


In [57]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
from collections import defaultdict, Counter

data = backup_cases_all

DARK  = '#333333'
LIGHT = '#aaaaaa'
BLUE  = '#2980b9'
RED   = '#cc3333'

years       = [int(c.split(']')[0].replace('[','').strip()) for c in data.keys()]
bench_sizes = [len(case.get('bench', [])) for case in data.values()]
judg_per_case = [len(case['judgmentAuthors']) for case in data.values()]
tags_per_case = [len(case.get('law_areas', [])) for case in data.values()]
para_per_case = [len(case.get('paragraphs', [])) for case in data.values()]
total_cites   = [sum(j.get('totalCitations', 0) for j in case['judgmentAuthors']) for case in data.values()]

year_counts = Counter(years)
year_list   = sorted(year_counts.keys())

# Tag frequency
from collections import defaultdict
tag_counter = Counter(tag for case in data.values() for tag in case.get('law_areas', []))

fig = plt.figure(figsize=(20, 22))
fig.patch.set_facecolor('white')
fig.suptitle(f'Dataset QC Dashboard  —  {len(data)} cases total (1997–2012)',
             fontsize=20, fontweight='bold', y=1.01)

gs = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

# ── 1. Cases per year ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.set_facecolor('#f8f8f8')
x = np.arange(len(year_list))
bars = ax1.bar(x, [year_counts[y] for y in year_list], color=DARK,
               edgecolor='white', linewidth=0.6, zorder=3)
for bar, y in zip(bars, year_list):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             str(year_counts[y]), ha='center', va='bottom', fontsize=9, color='#222')
ax1.set_xticks(x)
ax1.set_xticklabels(year_list, fontsize=10)
ax1.set_ylabel('Number of cases', fontsize=11)
ax1.set_title('Cases per year', fontsize=13, fontweight='bold', pad=10)
ax1.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax1.set_axisbelow(True)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── 2. Bench size distribution ─────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.set_facecolor('#f8f8f8')
bc = Counter(bench_sizes)
bx = sorted(bc.keys())
bars2 = ax2.bar(bx, [bc[b] for b in bx], color=DARK, edgecolor='white', linewidth=0.6, zorder=3)
for bar, val in zip(bars2, [bc[b] for b in bx]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', va='bottom', fontsize=9, color='#222')
ax2.set_xticks(bx)
ax2.set_xlabel('Bench size', fontsize=11)
ax2.set_ylabel('Number of cases', fontsize=11)
ax2.set_title('Bench size distribution', fontsize=13, fontweight='bold', pad=10)
ax2.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax2.set_axisbelow(True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# ── 3. Judgments per case ──────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_facecolor('#f8f8f8')
jc = Counter(judg_per_case)
jx = sorted(jc.keys())
ax3.bar(jx, [jc[j] for j in jx], color=DARK, edgecolor='white', linewidth=0.6, zorder=3)
for j_val in jx:
    ax3.text(j_val, jc[j_val] + 1, str(jc[j_val]), ha='center', va='bottom', fontsize=9, color='#222')
ax3.set_xticks(jx)
ax3.set_xlabel('Judgments per case', fontsize=11)
ax3.set_ylabel('Number of cases', fontsize=11)
ax3.set_title('Judgments per case', fontsize=13, fontweight='bold', pad=10)
ax3.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax3.set_axisbelow(True)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# ── 4. Tags per case ───────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_facecolor('#f8f8f8')
tc = Counter(tags_per_case)
tx = sorted(tc.keys())
ax4.bar(tx, [tc[t] for t in tx], color=DARK, edgecolor='white', linewidth=0.6, zorder=3)
for t_val in tx:
    ax4.text(t_val, tc[t_val] + 1, f'{tc[t_val]}\n({tc[t_val]/len(data):.0%})',
             ha='center', va='bottom', fontsize=8.5, color='#222')
ax4.set_xticks(tx)
ax4.set_xticklabels([f'{t} tag{"s" if t>1 else ""}' for t in tx])
ax4.set_ylabel('Number of cases', fontsize=11)
ax4.set_title('Tags per case', fontsize=13, fontweight='bold', pad=10)
ax4.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax4.set_axisbelow(True)
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

# ── 5. Paragraphs per case ─────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.set_facecolor('#f8f8f8')
pc = Counter(para_per_case)
px = sorted(pc.keys())
ax5.bar(px, [pc[p] for p in px], color=DARK, edgecolor='white', linewidth=0.6, zorder=3)
ax5.set_xlabel('Paragraphs per case', fontsize=11)
ax5.set_ylabel('Number of cases', fontsize=11)
ax5.set_title('Paragraphs per case', fontsize=13, fontweight='bold', pad=10)
stats_text = (f'Mean: {np.mean(para_per_case):.1f}\n'
              f'Median: {np.median(para_per_case):.0f}\n'
              f'Max: {max(para_per_case)}')
ax5.text(0.97, 0.97, stats_text, transform=ax5.transAxes, ha='right', va='top',
         fontsize=9, bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#ccc'))
ax5.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax5.set_axisbelow(True)
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)

# ── 6. Tag frequency ──────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, :2])
ax6.set_facecolor('#f8f8f8')
tags_sorted = sorted(tag_counter, key=lambda t: -tag_counter[t])
tag_vals    = [tag_counter[t] for t in tags_sorted]
tag_labels  = [t.replace(' & ', '\n& ').replace(', Planning', ',\nPlanning') for t in tags_sorted]
x6 = np.arange(len(tags_sorted))
ax6.bar(x6, tag_vals, color=DARK, edgecolor='white', linewidth=0.5, zorder=3)
for i, v in enumerate(tag_vals):
    ax6.text(i, v + 1, str(v), ha='center', va='bottom', fontsize=8, color='#222')
ax6.set_xticks(x6)
ax6.set_xticklabels(tag_labels, fontsize=7.5, rotation=40, ha='right')
ax6.set_ylabel('Number of cases', fontsize=11)
ax6.set_title('Tag frequency (all tags; cases may have multiple)', fontsize=13, fontweight='bold', pad=10)
ax6.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax6.set_axisbelow(True)
ax6.spines['top'].set_visible(False)
ax6.spines['right'].set_visible(False)

# ── 7. Total citations distribution (log scale) ────────────────────────
ax7 = fig.add_subplot(gs[2, 2])
ax7.set_facecolor('#f8f8f8')
ax7.hist([c for c in total_cites if c > 0], bins=30, color=DARK,
         edgecolor='white', linewidth=0.5, zorder=3)
ax7.set_xlabel('Total citations per case', fontsize=11)
ax7.set_ylabel('Number of cases', fontsize=11)
ax7.set_title('Total citation distribution', fontsize=13, fontweight='bold', pad=10)
ax7.set_yscale('log')
stats_text2 = (f'Mean: {np.mean(total_cites):.0f}\n'
               f'Median: {np.median(total_cites):.0f}\n'
               f'Max: {max(total_cites):,}')
ax7.text(0.97, 0.97, stats_text2, transform=ax7.transAxes, ha='right', va='top',
         fontsize=9, bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='#ccc'))
ax7.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax7.set_axisbelow(True)
ax7.spines['top'].set_visible(False)
ax7.spines['right'].set_visible(False)

plt.savefig('images/qc_dashboard.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [58]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
import numpy as np
from collections import defaultdict
from itertools import combinations

data = backup_cases_all 

co_authored  = defaultdict(int)
judge_totals = defaultdict(int)

for case in data.values():
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        for name in names:
            judge_totals[name] += 1
        for a, b in combinations(sorted(names), 2):
            co_authored[(a, b)] += 1

JUDGES = list(judge_totals.keys())

def build_graph(weight_fn):
    G = nx.Graph()
    G.add_nodes_from(JUDGES)
    for (a, b), count in co_authored.items():
        if a in JUDGES and b in JUDGES:
            w = weight_fn(a, b, count)
            if w > 0:
                G.add_edge(a, b, weight=w, raw=count)
    return G

cmap_edge = plt.cm.YlGnBu

def draw_network(G, title, filename, pos):
    weights = [G[u][v]['weight'] for u, v in G.edges()]
    w_min, w_max = min(weights), max(weights)

    fig, ax = plt.subplots(figsize=(12, 9))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.axis('off')

    # Draw edges with YlGnBu colormap
    for u, v in G.edges():
        w = G[u][v]['weight']
        t  = (w - w_min) / (w_max - w_min) if w_max > w_min else 1.0
        lw = 0.4 + t * 7.0
        color = cmap_edge(t)
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        ax.plot([x0, x1], [y0, y1], color=color, linewidth=lw,
                zorder=1, solid_capstyle='round', alpha=0.85)

    # Nodes
    max_total = max(judge_totals[j] for j in JUDGES)
    for judge in JUDGES:
        x, y   = pos[judge]
        size   = 300 + 1400 * (judge_totals[judge] / max_total)
        offset = 0.055 + 0.04 * (judge_totals[judge] / max_total)
        ax.scatter(x, y, s=size, color='#1c2833', zorder=3,
                   edgecolors='#555', linewidths=1.2)
        ax.text(x, y + offset, f"{judge.capitalize()} ({judge_totals[judge]})",
                ha='center', va='bottom', fontsize=9,
                color='#111', fontweight='bold', zorder=5)

    ax.set_title(title, fontsize=13, fontweight='bold', pad=14)

    # Colourbar legend
    sm = plt.cm.ScalarMappable(cmap=cmap_edge,
                                norm=mcolors.Normalize(vmin=w_min, vmax=w_max))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.01, location='left')
    cbar.set_label('Connection strength', fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"Saved: {filename}")

# ── Graph 1: raw — circular layout ───────────────────────────────────
G1 = build_graph(lambda a, b, count: count)

judges_sorted1 = sorted(JUDGES, key=lambda j: -judge_totals[j])
angle_step1    = 2 * np.pi / len(judges_sorted1)
pos1 = {j: (np.cos(i * angle_step1), np.sin(i * angle_step1))
        for i, j in enumerate(judges_sorted1)}
draw_network(G1,
    title="Judge Co-authorship Network (raw count)",
    filename='images/judge_network_raw.png',
    pos=pos1)

# ── Graph 2: normalised — circular layout ─────────────────────────────
def norm_weight(a, b, count):
    return (2 * count) / (judge_totals[a] + judge_totals[b])

G2 = build_graph(norm_weight)

# Circular layout with judges sorted by total judgments
judges_sorted = sorted(JUDGES, key=lambda j: -judge_totals[j])
angle_step = 2 * np.pi / len(judges_sorted)
pos2 = {j: (np.cos(i * angle_step), np.sin(i * angle_step))
        for i, j in enumerate(judges_sorted)}

draw_network(G2,
    title="Judge Co-authorship Network (normalised by individual output)",
    filename='images/judge_network_normalised.png',
    pos=pos2)

Saved: images/judge_network_raw.png
Saved: images/judge_network_normalised.png


In [59]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

data = backup_cases_all

TOP100 = ['[1998] HCA 28','[2003] HCA 2','[2003] HCA 22','[2001] HCA 30','[2009] HCA 27','[2005] HCA 25','[2010] HCA 16','[1998] HCA 11','[2000] HCA 63','[2006] HCA 63','[1999] HCA 14','[2001] HCA 17','[2000] HCA 54','[1998] HCA 57','[2010] HCA 45','[2004] HCA 52','[2009] HCA 41','[2001] HCA 64','[1999] HCA 21','[2000] HCA 57','[2011] HCA 39','[2000] HCA 47','[2010] HCA 28','[2007] HCA 22','[2010] HCA 1','[2005] HCA 24','[2003] HCA 6','[2000] HCA 48','[2006] HCA 46','[2010] HCA 4','[1998] HCA 67','[1999] HCA 54','[2000] HCA 40','[2008] HCA 31','[1999] HCA 29','[2011] HCA 49','[2011] HCA 4','[2002] HCA 11','[2001] HCA 63','[2004] HCA 35','[2002] HCA 53','[2006] HCA 27','[2000] HCA 41','[2002] HCA 36','[1999] HCA 27','[1998] HCA 27','[1999] HCA 26','[1999] HCA 66','[2001] HCA 22','[1999] HCA 67','[2002] HCA 35','[2011] HCA 48','[2011] HCA 13','[2010] HCA 41','[1999] HCA 9','[1998] HCA 55','[1998] HCA 30','[2011] HCA 21','[1999] HCA 36','[2007] HCA 35','[2005] HCA 12','[2011] HCA 1','[2005] HCA 81','[2008] HCA 36','[1999] HCA 18','[2000] HCA 5','[2010] HCA 23','[2004] HCA 45','[2001] HCA 52','[2000] HCA 12','[2001] HCA 44','[2009] HCA 25','[1999] HCA 10','[2002] HCA 49','[2011] HCA 34','[2001] HCA 59','[2002] HCA 6','[2003] HCA 71','[2011] HCA 11','[1998] HCA 69','[2001] HCA 29','[2007] HCA 30','[2010] HCA 48','[2002] HCA 46','[1997] HCA 56','[2002] HCA 54','[2004] HCA 60','[2000] HCA 3','[2005] HCA 10','[2010] HCA 32','[2001] HCA 18','[1999] HCA 37','[2002] HCA 28','[1999] HCA 6','[1998] HCA 1','[2004] HCA 46','[2005] HCA 62','[2006] HCA 55','[2011] HCA 10','[2004] HCA 37','[2003] HCA 1']

judge_rc       = defaultdict(list)
gh_together_rc = []
gummow_rc      = []
hayne_rc       = []

for citation in TOP100:
    if citation not in data:
        continue
    case = data[citation]
    for j in case['judgmentAuthors']:
        names = [n.strip() for n in j['name'].split(',')]
        rc    = j.get('totalCitations', 0)
        has_g = 'GUMMOW' in names
        has_h = 'HAYNE'  in names
        if has_g and has_h:
            gh_together_rc.append(rc)
        if has_g:
            gummow_rc.append(rc)
        if has_h:
            hayne_rc.append(rc)
        for name in names:
            if name not in ('GUMMOW', 'HAYNE'):
                judge_rc[name].append(rc)

JUDGES = sorted(judge_rc.keys())
GH    = '#1a5276'
GBLUE = '#2980b9'
DARK  = '#333333'
RED   = '#cc3333'

all_panels = [
    ('Gummow\n& Hayne', gh_together_rc, GH),
    ('Gummow',           gummow_rc,      GBLUE),
    ('Hayne',            hayne_rc,       GBLUE),
] + [(j.capitalize(), judge_rc[j], DARK) for j in JUDGES]

n_total = len(all_panels)
n_cols  = (n_total + 1) // 2
fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 12), sharey=True)
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

def draw_box(ax, vals, color, title):
    ax.set_facecolor('#f8f8f8')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

    bp = ax.boxplot(
        [vals], positions=[1], widths=0.5, patch_artist=True,
        medianprops=dict(color=RED, linewidth=2),
        whiskerprops=dict(color='#555'),
        capprops=dict(color='#555'),
        flierprops=dict(marker='o', markersize=2.5, alpha=0.35,
                        markerfacecolor='#999', markeredgecolor='none'),
    )
    bp['boxes'][0].set_facecolor(color)
    bp['boxes'][0].set_alpha(0.8)

    ax.set_xticks([1])
    ax.set_xticklabels([''], fontsize=8.5)
    ax.set_xlim(0.5, 1.5)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.text(1, -0.04, f'n={len(vals)}', ha='center', va='top',
            fontsize=8, color='#777', transform=ax.get_xaxis_transform())
    ax.text(1, -0.10, f'med={np.median(vals):.0f}', ha='center', va='top',
            fontsize=8, color='#333', fontweight='bold',
            transform=ax.get_xaxis_transform())

for i, (title, vals, color) in enumerate(all_panels):
    draw_box(axes_flat[i], vals, color, title)

for j in range(len(all_panels), len(axes_flat)):
    axes_flat[j].set_visible(False)

for row in range(2):
    axes[row][0].set_ylabel('Raw citations', fontsize=11)

fig.suptitle('Raw Citation Count: G+H together vs. each other judge\n(top 99 most cited cases)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/citation_raw_top100.png', dpi=150,
            bbox_inches='tight', facecolor='white')
print("Saved.")

Saved.


In [ ]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
from collections import defaultdict

data = backup_cases_all

# Only cases where BOTH were on the bench
yearly = defaultdict(lambda: {'together': 0, 'bench': 0})

for citation, case in data.items():
    year = int(citation.split(']')[0].replace('[', '').strip())
    bench = case.get('bench', [])
    if 'GUMMOW' not in bench or 'HAYNE' not in bench:
        continue
    yearly[year]['bench'] += 1
    if any('GUMMOW' in [n.strip() for n in j['name'].split(',')]
           and 'HAYNE' in [n.strip() for n in j['name'].split(',')]
           for j in case['judgmentAuthors']):
        yearly[year]['together'] += 1

years  = sorted(y for y in yearly if y >= 1998)
tog    = [yearly[y]['together'] for y in years]
bench  = [yearly[y]['bench']    for y in years]
not_t  = [b - t for b, t in zip(bench, tog)]
rate   = [t / b for t, b in zip(tog, bench)]

BAR_DARK  = '#333333'
BAR_LIGHT = '#aaaaaa'
LINE_COLOR = '#cc3333'

fig, ax1 = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('white')
ax1.set_facecolor('#f8f8f8')

x = np.arange(len(years))
ax1.bar(x, bench, color=BAR_LIGHT, edgecolor='white', linewidth=0.6, zorder=3, label='On same bench (did not write together)')
ax1.bar(x, tog,   color=BAR_DARK,  edgecolor='white', linewidth=0.6, zorder=4, label='G+H wrote together')

for i, (t, b) in enumerate(zip(tog, bench)):
    ax1.text(i, b + 0.4, str(b), ha='center', va='bottom', fontsize=8, color='#555')

ax1.set_xticks(x)
ax1.set_xticklabels(years, fontsize=10)
ax1.set_ylabel('Number of cases', fontsize=11)
ax1.set_xlabel('Year', fontsize=11)
ax1.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax1.set_axisbelow(True)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2 = ax1.twinx()
ax2.plot(x, rate, color=LINE_COLOR, linewidth=2, marker='o',
         markersize=5, zorder=5, label='Co-authorship rate')
ax2.set_ylabel('Co-authorship rate', fontsize=11, color=LINE_COLOR)
ax2.tick_params(axis='y', labelcolor=LINE_COLOR)
ax2.set_ylim(0, 1)
ax2.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax2.spines['top'].set_visible(False)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           fontsize=10, frameon=True, facecolor='white', edgecolor='#aaa', loc='upper left')

ax1.set_title('Gummow & Hayne — Co-authorship over time\n(cases where both were on the bench)',
              fontsize=13, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('images/gh_over_time_bench.png', dpi=150, bbox_inches='tight', facecolor='white')
print("Saved.")